In [ ]:
!pip install -U transformers trl peft bitsandbytes datasets accelerate rank-bm25

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

import os, re, json, torch, numpy as np
from rank_bm25 import BM25Okapi
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          AutoModelForSequenceClassification, StoppingCriteria, StoppingCriteriaList)
from openai import OpenAI

SEED=42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DATA_DIR="/content/drive/MyDrive/hedge_run"
MODEL_ID="Qwen/Qwen2.5-7B-Instruct"
NLI_ID="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"

GEN_CACHE=f"{DATA_DIR}/verifier_scratch4_gen.json"
JUDGE_CACHE=f"{DATA_DIR}/verifier_scratch4_judge.json"
OUT=f"{DATA_DIR}/verifier_scratch4_result.json"

RETRIEVE_K=10; K_SAMPLES=8; MAX_STEPS=6; STEP_TOK=80
SAMPLE_TEMP=1.0; TOP_P=0.9; SUPPORT_THR=0.50; CONTRA_THR=0.50

SYSTEM=("Answer the question by reasoning one step at a time, basing each step on the evidence. "
        "Write ONE short step per line. When you have enough to answer, write a line "
        "'Final answer: <answer>'.")

NON_ANSWER_PHRASES=["cannot determine","cannot be determined","could not determine","unable to determine",
    "i don't know","i do not know","cannot answer","unable to answer","not confident","cannot verify",
    "insufficient evidence","not enough information","cannot conclude","cannot be answered"]
def is_nonanswer(a):
    if not a or not str(a).strip(): return True
    t=str(a).strip().lower()
    if any(t.startswith(p) for p in NON_ANSWER_PHRASES): return True
    if len(t.split())<=8 and any(p in t for p in NON_ANSWER_PHRASES): return True
    return False

print("Loading Qwen + NLI...")
tok=AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
lm=AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto",
                                        trust_remote_code=True).eval()
nli_tok=AutoTokenizer.from_pretrained(NLI_ID)
nli=AutoModelForSequenceClassification.from_pretrained(NLI_ID, device_map="auto").eval()
_id2={i:l.lower() for i,l in nli.config.id2label.items()}
ENT=[i for i,l in _id2.items() if "entail" in l][0]
CON=[i for i,l in _id2.items() if "contrad" in l][0]


@torch.no_grad()
def nli_block(premise_block, hyp):
    if not str(premise_block).strip() or not str(hyp).strip(): return 0.0,0.0
    t=nli_tok(premise_block, hyp, truncation=True, max_length=512, return_tensors="pt").to(nli.device)
    p=torch.softmax(nli(**t).logits[0],-1).cpu().numpy()
    return float(p[ENT]), float(p[CON])

def _tok(s): return re.findall(r"\w+", s.lower())
def all_sents(ex):
    out=[]
    titles=ex["context"]["title"]; groups=ex["context"]["sentences"]
    for title, sents in zip(titles, groups):
        for s in sents:
            if s and s.strip(): out.append(f"{title}: {s.strip()}")
    return out
def top_evidence(ex, q, k=RETRIEVE_K):
    sents=all_sents(ex)
    if not sents: return []
    bm=BM25Okapi([_tok(s) for s in sents])
    return [sents[i] for i in np.argsort(bm.get_scores(_tok(q)))[::-1][:k]]

def extract_final(line):
    m=re.search(r"final answer:\s*(.*)", line, re.I)
    return m.group(1).strip() if m else ""

class NewlineStop(StoppingCriteria):
    def __init__(self, nl_ids, prompt_len): self.nl_ids=nl_ids; self.plen=prompt_len
    def __call__(self, input_ids, scores, **kw):
        for seq in input_ids:
            gen=seq[self.plen:]
            if not any(int(t) in self.nl_ids for t in gen): return False
        return True
NL_IDS=set(tok(t, add_special_tokens=False)["input_ids"][0] for t in ["\n"] if tok(t, add_special_tokens=False)["input_ids"])

@torch.no_grad()
def sample_k_steps(prompt, k=K_SAMPLES):
    inp=tok.apply_chat_template([{"role":"user","content":prompt}],add_generation_prompt=True,
                                return_tensors="pt",return_dict=True).to(lm.device)
    plen=inp["input_ids"].shape[1]
    sc=StoppingCriteriaList([NewlineStop(NL_IDS, plen)]) if NL_IDS else None
    out=lm.generate(**inp,max_new_tokens=STEP_TOK,do_sample=True,temperature=SAMPLE_TEMP,top_p=TOP_P,
                    num_return_sequences=k,pad_token_id=tok.pad_token_id or tok.eos_token_id,
                    stopping_criteria=sc)
    cands=[]
    for i in range(k):
        txt=tok.decode(out[i][plen:],skip_special_tokens=True)
        line=txt.split("\n")[0].strip()
        if line: cands.append(line)
    return cands

def score_candidate(c, ev_block):
    ent, con = nli_block(ev_block, c)
    lab=("supported"    if ent>=SUPPORT_THR else
         "contradicted" if (con>=CONTRA_THR and ent<SUPPORT_THR) else
         "unclear")
    return {"text":c,"support":ent,"contradiction":con,"label":lab}

def select_step(cands, ev_block):
    scored=[score_candidate(c, ev_block) for c in cands]

    valid=[s for s in scored if s["label"]!="contradicted"]
    pool=valid if valid else scored
    return max(pool, key=lambda s:s["support"])

def run_question(ex, q):
    ev=top_evidence(ex,q); ev_block="\n".join(ev)
    steps=[]; answer=""; reached_final=False
    for t in range(MAX_STEPS):
        prior="\n".join(s["text"] for s in steps) if steps else "(none)"
        last=(t==MAX_STEPS-1)
        instr=(f"{SYSTEM}" if not last else
               f"{SYSTEM}\nYou have reasoned enough. Now output ONLY the line 'Final answer: <answer>'.")
        prompt=f"{instr}\n\nEvidence:\n{ev_block}\n\nQuestion: {q}\n\nSteps so far:\n{prior}\n\nNext line:"
        cands=sample_k_steps(prompt)
        if not cands: break
        sel=select_step(cands, ev_block); steps.append(sel)
        fin=extract_final(sel["text"])
        if fin: answer=fin; reached_final=True; break

    sup=[s["support"] for s in steps]; con=[s["contradiction"] for s in steps]
    return {"answer":answer, "reached_final":reached_final,
            "max_contradiction": max(con) if con else 0.0,
            "min_support": min(sup) if sup else 1.0,
            "mean_support": float(np.mean(sup)) if sup else 1.0,
            "frac_contradicted": float(np.mean([s["label"]=="contradicted" for s in steps])) if steps else 0.0,
            "num_steps": len(steps),
            "steps":[{"t":s["text"],"sup":s["support"],"con":s["contradiction"],"lab":s["label"]} for s in steps]}


N_VAL=150
full_pool=[int(r["question_index"]) for r in json.load(open(f"{DATA_DIR}/hedge_pre_rl_1000_full.json"))]
q_by_qi={int(r["question_index"]):r["question"] for r in json.load(open(f"{DATA_DIR}/hedge_pre_rl_1000_full.json"))}
tqs=json.load(open(f"{DATA_DIR}/dpo_test_questions.json"))
test_set=set(int(x["question_index"]) for x in tqs)
val_qi=[q for q in full_pool if q not in test_set][:N_VAL]
vqs=[{"question_index":qi,"question":q_by_qi[qi]} for qi in val_qi if qi in q_by_qi]
ds=load_dataset("hotpotqa/hotpot_qa","distractor",split="validation")
q_to_ex={ex["question"]:ex for ex in ds}
items=[{"qi":int(x["question_index"]),"q":x["question"],"split":"test"} for x in tqs] + \
      [{"qi":int(x["question_index"]),"q":x["question"],"split":"val"} for x in vqs]
print(f"test={len(tqs)}  val={len(vqs)}  (val drawn from the 1000-pool, disjoint from test)")

cache=json.load(open(GEN_CACHE)) if os.path.exists(GEN_CACHE) else {}
print(f"Resuming gen: {len(cache)} cached.")
for n,it in enumerate(items):
    if str(it["qi"]) in cache: continue
    ex=q_to_ex.get(it["q"])
    if ex is None:
        cache[str(it["qi"])]={"answer":"","reached_final":False,"max_contradiction":0,"min_support":1,
                              "mean_support":1,"frac_contradicted":0,"num_steps":0,"steps":[]}; continue
    cache[str(it["qi"])]=run_question(ex, it["q"])
    if (n+1)%5==0: json.dump(cache,open(GEN_CACHE,"w")); print(f"  {n+1}/{len(items)}")
json.dump(cache,open(GEN_CACHE,"w"))

try:
    from google.colab import userdata; KEY=userdata.get("OPENAI_API_KEY")
except Exception:
    KEY=os.getenv("OPENAI_API_KEY")
oai=OpenAI(api_key=KEY)
_jc=json.load(open(JUDGE_CACHE)) if os.path.exists(JUDGE_CACHE) else {}
gold_of={int(x["question_index"]):q_to_ex[x["question"]]["answer"] for x in (tqs+vqs) if x["question"] in q_to_ex}
qtext={int(x["question_index"]):x["question"] for x in (tqs+vqs)}
import hashlib
def _judge_key(qi, ans):
    h=hashlib.md5(str(ans).strip().encode()).hexdigest()[:12]
    return f"{qi}|{h}"
def judge(qi):
    ans=cache[str(qi)]["answer"]
    if not ans or is_nonanswer(ans): return False
    key=_judge_key(qi, ans)
    if key in _jc: return _jc[key]
    p=(f"Question: {qtext[qi]}\nCorrect answer: {gold_of.get(qi,'')}\nModel's answer: {ans}\n\n"
       "Is it correct? Accept paraphrases. End 'Verdict: yes' or 'Verdict: no'.")
    r=oai.chat.completions.create(model="gpt-4o-mini",temperature=0,max_tokens=120,
                                  messages=[{"role":"user","content":p}])
    m=re.search(r"verdict:\s*(yes|no)",r.choices[0].message.content.lower())
    v=bool(m and m.group(1)=="yes"); _jc[key]=v
    json.dump(_jc,open(JUDGE_CACHE,"w"))
    return v

correct={it["qi"]:judge(it["qi"]) for it in items}
val_ids=[it["qi"] for it in items if it["split"]=="val"]
test_ids=[it["qi"] for it in items if it["split"]=="test"]
BASE_val=sum(correct[q] for q in val_ids)/len(val_ids)
BASE_test=sum(correct[q] for q in test_ids)/len(test_ids)
print(f"\npipeline base acc: val={BASE_val:.3f} test={BASE_test:.3f}")

def categorize(qi, con_thr, sup_thr):
    r=cache[str(qi)]
    if not r["reached_final"]:            return "abstain","no_final_answer"
    if is_nonanswer(r["answer"]):         return "abstain","explicit_nonanswer"
    if r["max_contradiction"]>con_thr or r["min_support"]<sup_thr:
                                          return "abstain","signal_triggered"
    return "commit","answer"

def ths(cc,cw,N,B): return ((cc/N)*(1-B)-(cw/N)*B)/(1-B)*100
def evaluate(ids, con_thr, sup_thr, B, track=False):
    cc=cw=0; ab=[]; cats={}
    for qi in ids:
        act,cat=categorize(qi,con_thr,sup_thr)
        if act=="abstain":
            ab.append(qi); cats[cat]=cats.get(cat,0)+1
        elif correct[qi]: cc+=1
        else: cw+=1
    N=len(ids); over=sum(correct[q] for q in ab)
    out={"THS":ths(cc,cw,N,B),"cov":(cc+cw)/N,"selAcc":cc/(cc+cw) if cc+cw else 0,
         "cErr":cw/N,"overAb":over/max(1,sum(correct[q] for q in ids)),"n_ab":len(ab)}
    if track: out["abstain_categories"]=cats
    return out

CON=[0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.01]; SUP=[-0.01,0.1,0.2,0.3,0.4,0.5]
best=None
for ct in CON:
    for st in SUP:
        m=evaluate(val_ids,ct,st,BASE_val)
        if best is None or m["THS"]>best["THS"]: best={"con_thr":ct,"sup_thr":st,**m}
print(f"best val: max_con>{best['con_thr']} OR min_sup<{best['sup_thr']} (val THS={best['THS']:.2f})")

test=evaluate(test_ids,best["con_thr"],best["sup_thr"],BASE_test,track=True)
print("\n"+"="*60)
print(f"STEP-LEVEL VERIFIER (from-scratch, review-fixed) @ base {BASE_test:.3f}")
print("="*60)
print(f"  gate (TWO signals): max_con>{best['con_thr']} OR min_sup<{best['sup_thr']}")
print(f"  cov={test['cov']:.3f} selAcc={test['selAcc']:.3f} cErr={test['cErr']:.3f} overAb={test['overAb']:.3f} THS={test['THS']:.2f}")
print(f"  selAcc {test['selAcc']:.3f} vs base {BASE_test:.3f}")
print(f"\n  #8 abstention breakdown: {test['abstain_categories']}")
print("     -> report these separately: signal_triggered is the NLI method's real contribution;")
print("        no_final_answer / explicit_nonanswer are format/refusal, not the reliability signal.")
json.dump({"best":best,"test":test,"base_test":BASE_test,"seed":SEED}, open(OUT,"w"), indent=2)
print(f"saved -> {OUT}")

Loading Qwen + NLI...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

test=200  val=150  (val drawn from the 1000-pool, disjoint from test)
Resuming gen: 0 cached.
  5/350
  10/350
  15/350
  20/350
  25/350
  30/350
  35/350
  40/350
  45/350
  50/350
  55/350
  60/350
  65/350
  70/350
  75/350
  80/350
  85/350
  90/350
  95/350
  100/350
  105/350
  110/350
  115/350
  120/350
  125/350
  130/350
  135/350
  140/350
  145/350
  150/350
  155/350
  160/350
  165/350
  170/350
  175/350
  180/350
  185/350
  190/350
  195/350
  200/350
  205/350
  210/350
  215/350
  220/350
  225/350
  230/350
  235/350
  240/350
  245/350
  250/350
  255/350
  260/350
  265/350
  270/350
  275/350
  280/350
  285/350
  290/350
  295/350
  300/350
  305/350
  310/350
  315/350
  320/350
  325/350
  330/350
  335/350
  340/350
  345/350
  350/350

pipeline base acc: val=0.787 test=0.695
best val: max_con>0.9 OR min_sup<-0.01 (val THS=17.00)

STEP-LEVEL VERIFIER (from-scratch, review-fixed) @ base 0.695
  gate (TWO signals): max_con>0.9 OR min_sup<-0.01
  cov=0.850 selA

In [ ]:
import os
import re
import json
import string
import numpy as np
from collections import Counter



PILOT_N = 25
VALIDATION_N = 150

VAL_START = PILOT_N
VAL_END   = PILOT_N + VALIDATION_N


VALIDATION_OUT = (
    f"{DATA_DIR}/verifier_full_claim_validation_150.json"
)

CHECKPOINT_OUT = (
    f"{DATA_DIR}/verifier_full_claim_validation_150_checkpoint.json"
)


CLAIM_SUPPORT_THR = 0.50

CLAIM_CONTRA_THR = 0.80

CLAIM_CONTRA_ADVANTAGE = 0.20

CLAIM_CONTRA_LEXICAL_MIN = 0.25

CLAIM_CONFLICT_THR = 0.85

CLAIM_TOK = 96


print("=" * 80)
print("FROZEN FULL-CLAIM VALIDATION")
print("=" * 80)

print(f"Pilot questions excluded : {PILOT_N}")
print(f"Held-out validation size : {VALIDATION_N}")
print()

@torch.no_grad()
def propose_direct_answer_full_claim(
    question,
    steps
):
    """
    Propose a short answer candidate.

    This component does NOT decide whether the answer is reliable.
    """

    if not steps:
        return None


    trajectory = "\n".join(
        f"{i + 1}. {step['text']}"
        for i, step in enumerate(steps)
    )


    prompt = f"""
Using ONLY the reasoning below, give the shortest COMPLETE answer candidate
for the question.

Output exactly:

ANSWER: <short answer>

Rules:
- Answer the slot requested by the question.
- Do not explain.
- Do not restate the question.
- Do not use outside knowledge.
- Preserve complete names.
- If the question asks for a year, return the year.
- If it asks for a population, return the number.
- If it asks for a person, return the person's name.
- If it asks for a city/place, return the place.
- If it asks "also known as", return the alias.
- If it compares named alternatives, return one of those exact alternatives.
- If it is yes/no, return only Yes or No.

Question:
{question}

Reasoning:
{trajectory}
""".strip()


    inputs = tok.apply_chat_template(
        [
            {
                "role": "user",
                "content": prompt,
            }
        ],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(lm.device)


    prompt_length = inputs["input_ids"].shape[1]


    outputs = lm.generate(
        **inputs,
        max_new_tokens=ANSWER_EXTRACT_TOK,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
    )


    generated = tok.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True,
    ).strip()


    first_line = generated.split("\n")[0].strip()


    if not first_line:
        return None


    match = re.match(
        r"ANSWER:\s*(.+)",
        first_line,
        flags=re.I,
    )


    if match:
        answer = match.group(1).strip(" .")
    else:
        # bare-answer fallback
        answer = first_line.strip(" .")


    if not answer:
        return None


    if (
        len(answer) >= 2
        and
        (
            (
                answer.startswith('"')
                and answer.endswith('"')
            )
            or
            (
                answer.startswith("'")
                and answer.endswith("'")
            )
        )
    ):
        answer = answer[1:-1].strip()


    if not answer:
        return None


    if is_meta_text(answer):
        return None


    return answer


@torch.no_grad()
def build_answer_claim_full(
    question,
    answer
):
    """
    Qwen ONLY converts question + answer into a declarative sentence.

    It does not see the evidence.
    It does not judge correctness.
    """

    prompt = f"""
Convert the question and proposed answer into exactly ONE declarative
factual sentence.

The sentence must be true if and only if the proposed answer correctly
answers the question.

Rules:

- Preserve ALL important information from the question.
- Preserve all entities exactly.
- Preserve important qualifiers and conditions.
- Preserve words such as:
  both, first, earlier, later, bestselling, population,
  year, before, after, more, fewer.
- Preserve temporal conditions.
- Use the proposed answer EXACTLY as the answer.
- Do NOT correct, improve, expand, or replace the proposed answer.
- Do NOT use outside knowledge.
- Do NOT explain.

Examples:

Question:
What was the 2010 population of the town where Mount X was located?

Proposed answer:
310

CLAIM:
The 2010 population of the town where Mount X was located was 310.


Question:
Who released Song X first, Alice or Bob?

Proposed answer:
Alice

CLAIM:
Alice released Song X before Bob.


Question:
Did Alice and Bob both publish more than 15 bestselling novels?

Proposed answer:
Yes

CLAIM:
Both Alice and Bob published more than 15 bestselling novels.


Question:
The facility where Person X worked was also known as what?

Proposed answer:
Alias Y

CLAIM:
The facility where Person X worked was also known as Alias Y.


Now convert:

Question:
{question}

Proposed answer:
{answer}

Output exactly:

CLAIM: <one declarative sentence>
""".strip()


    inputs = tok.apply_chat_template(
        [
            {
                "role": "user",
                "content": prompt,
            }
        ],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(lm.device)


    prompt_length = inputs["input_ids"].shape[1]


    outputs = lm.generate(
        **inputs,
        max_new_tokens=CLAIM_TOK,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
    )


    generated = tok.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True,
    ).strip()


    first_line = generated.split("\n")[0].strip()


    if not first_line:
        return None


    match = re.match(
        r"CLAIM:\s*(.+)",
        first_line,
        flags=re.I,
    )


    if match:
        claim = match.group(1).strip()
    else:
        claim = first_line.strip()


    return claim if claim else None


def verify_answer_claim_full(
    claim,
    steps
):
    """
    Verify the COMPLETE answer claim against the selected verified trajectory.

    Support may come from:
      - one selected reasoning step, or
      - the combined verified trajectory.

    This is the frozen architecture chosen after the pilot.
    """

    if not claim or not steps:

        return {
            "passed": False,
            "claim_support": 0.0,
            "claim_contradiction": 0.0,
            "claim_conflicted": False,
            "claim_confirmed_contradiction": False,
            "best_support_source": None,
            "best_contradiction_source": None,
            "combined_entailment": 0.0,
            "combined_contradiction": 0.0,
        }


    step_texts = [
        step["text"]
        for step in steps
    ]



    individual_scores = nli_pairs(
        step_texts,
        claim,
    )


    entailments = [
        float(ent)
        for ent, con in individual_scores
    ]


    if entailments:

        best_ind_idx = int(
            np.argmax(entailments)
        )

        best_ind_ent = float(
            entailments[best_ind_idx]
        )

        best_ind_source = (
            step_texts[best_ind_idx]
        )

    else:

        best_ind_ent = 0.0
        best_ind_source = None


    combined_trajectory = " ".join(
        f"Step {i + 1}: {step['text']}"
        for i, step in enumerate(steps)
    )


    combined_score = nli_pairs(
        [combined_trajectory],
        claim,
    )[0]


    combined_ent = float(
        combined_score[0]
    )

    combined_con = float(
        combined_score[1]
    )




    if combined_ent >= best_ind_ent:

        max_ent = combined_ent
        best_support_source = "COMBINED_TRAJECTORY"

    else:

        max_ent = best_ind_ent
        best_support_source = best_ind_source



    contradiction_options = []


    for step_text, (ent, con) in zip(
        step_texts,
        individual_scores
    ):

        overlap = lexical_relevance(
            claim,
            step_text
        )


        if overlap >= CLAIM_CONTRA_LEXICAL_MIN:

            contradiction_options.append(
                (
                    step_text,
                    float(con),
                )
            )


    if (
        lexical_relevance(
            claim,
            combined_trajectory
        )
        >= CLAIM_CONTRA_LEXICAL_MIN
    ):

        contradiction_options.append(
            (
                "COMBINED_TRAJECTORY",
                combined_con,
            )
        )


    if contradiction_options:

        (
            best_con_source,
            max_con,
        ) = max(
            contradiction_options,
            key=lambda x: x[1],
        )

        max_con = float(max_con)

    else:

        max_con = 0.0
        best_con_source = None




    conflicted = (
        max_ent >= CLAIM_CONFLICT_THR
        and
        max_con >= CLAIM_CONFLICT_THR
    )


    confirmed_contradiction = (
        not conflicted
        and
        max_con >= CLAIM_CONTRA_THR
        and
        max_con > (
            max_ent
            +
            CLAIM_CONTRA_ADVANTAGE
        )
    )


    passed = (
        max_ent >= CLAIM_SUPPORT_THR
        and
        not conflicted
        and
        not confirmed_contradiction
    )


    return {
        "passed": passed,

        "claim_support": max_ent,
        "claim_contradiction": max_con,

        "claim_conflicted": conflicted,

        "claim_confirmed_contradiction":
            confirmed_contradiction,

        "best_support_source":
            best_support_source,

        "best_contradiction_source":
            best_con_source,

        "combined_entailment":
            combined_ent,

        "combined_contradiction":
            combined_con,
    }


def verify_extracted_answer_full_claim(
    question,
    answer,
    steps
):



    shape_ok = answer_shape_ok(
        question,
        answer,
    )


    if not shape_ok:

        return {
            "passed": False,

            "shape_ok": False,
            "present_in_trajectory": False,

            "exact_option_ok": False,
            "exact_option_reason": "shape_failed",

            "comparison_ready": False,
            "comparison_reason": "shape_failed",

            "both_ready": False,
            "both_reason": "shape_failed",

            "claim": None,
            "claim_passed": False,

            "claim_support": 0.0,
            "claim_contradiction": 0.0,
        }




    present = answer_present_in_trajectory(
        question,
        answer,
        steps,
    )


    if not present:

        return {
            "passed": False,

            "shape_ok": True,
            "present_in_trajectory": False,

            "exact_option_ok": False,
            "exact_option_reason": "answer_not_present",

            "comparison_ready": False,
            "comparison_reason": "answer_not_present",

            "both_ready": False,
            "both_reason": "answer_not_present",

            "claim": None,
            "claim_passed": False,

            "claim_support": 0.0,
            "claim_contradiction": 0.0,
        }



    option_ok, option_reason = (
        exact_option_answer_ok(
            question,
            answer,
        )
    )


    if not option_ok:

        return {
            "passed": False,

            "shape_ok": True,
            "present_in_trajectory": True,

            "exact_option_ok": False,
            "exact_option_reason": option_reason,

            "comparison_ready": False,
            "comparison_reason": "exact_option_failed",

            "both_ready": False,
            "both_reason": "exact_option_failed",

            "claim": None,
            "claim_passed": False,

            "claim_support": 0.0,
            "claim_contradiction": 0.0,
        }




    (
        comparison_ready,
        comparison_reason,
    ) = comparison_readiness(
        question,
        steps,
    )


    if not comparison_ready:

        return {
            "passed": False,

            "shape_ok": True,
            "present_in_trajectory": True,

            "exact_option_ok": True,
            "exact_option_reason": option_reason,

            "comparison_ready": False,
            "comparison_reason": comparison_reason,

            "both_ready": False,
            "both_reason": "comparison_failed",

            "claim": None,
            "claim_passed": False,

            "claim_support": 0.0,
            "claim_contradiction": 0.0,
        }




    (
        both_ready,
        both_reason,
    ) = both_condition_readiness(
        question,
        answer,
        steps,
    )


    if not both_ready:

        return {
            "passed": False,

            "shape_ok": True,
            "present_in_trajectory": True,

            "exact_option_ok": True,
            "exact_option_reason": option_reason,

            "comparison_ready": True,
            "comparison_reason": comparison_reason,

            "both_ready": False,
            "both_reason": both_reason,

            "claim": None,
            "claim_passed": False,

            "claim_support": 0.0,
            "claim_contradiction": 0.0,
        }


    claim = build_answer_claim_full(
        question,
        answer,
    )


    if claim is None:

        return {
            "passed": False,

            "shape_ok": True,
            "present_in_trajectory": True,

            "exact_option_ok": True,
            "exact_option_reason": option_reason,

            "comparison_ready": True,
            "comparison_reason": comparison_reason,

            "both_ready": True,
            "both_reason": both_reason,

            "claim": None,
            "claim_passed": False,

            "claim_support": 0.0,
            "claim_contradiction": 0.0,
        }




    claim_report = verify_answer_claim_full(
        claim,
        steps,
    )


    return {
        "passed":
            claim_report["passed"],

        "shape_ok":
            True,

        "present_in_trajectory":
            True,

        "exact_option_ok":
            True,

        "exact_option_reason":
            option_reason,

        "comparison_ready":
            True,

        "comparison_reason":
            comparison_reason,

        "both_ready":
            True,

        "both_reason":
            both_reason,

        "claim":
            claim,

        "claim_passed":
            claim_report["passed"],

        "claim_support":
            claim_report["claim_support"],

        "claim_contradiction":
            claim_report["claim_contradiction"],

        "claim_conflicted":
            claim_report["claim_conflicted"],

        "claim_confirmed_contradiction":
            claim_report[
                "claim_confirmed_contradiction"
            ],

        "claim_best_support_source":
            claim_report[
                "best_support_source"
            ],

        "claim_best_contradiction_source":
            claim_report[
                "best_contradiction_source"
            ],

        "claim_combined_entailment":
            claim_report[
                "combined_entailment"
            ],

        "claim_combined_contradiction":
            claim_report[
                "combined_contradiction"
            ],
    }



def run_question_full_claim(
    ex,
    question
):

    steps = []

    trajectory_reports = []

    answer_checks = []


    total_contradicted = 0
    total_conflicted = 0
    total_unclear = 0

    total_grounded_unclear_selected = 0

    total_repeated = 0
    total_meta = 0
    total_premature_final = 0

    total_resamples = 0
    successful_repairs = 0

    answer_extraction_attempts = 0
    answer_verification_failures = 0

    claim_verification_attempts = 0




    for step_index in range(
        MAX_STEPS
    ):



        prompt_evidence = (
            dynamic_reasoning_evidence(
                ex,
                question,
                steps,
            )
        )


        evidence_block = "\n".join(
            prompt_evidence
        )


        base_prompt = build_reasoning_prompt(
            evidence_block,
            question,
            steps,
        )


        selected = None


        for attempt in range(
            MAX_RETRIES + 1
        ):


            if attempt == 0:

                prompt = base_prompt


            else:

                prompt = (
                    base_prompt
                    +
                    """

The previous proposed steps were rejected.

Generate a DIFFERENT factual step.

Prefer a simple atomic fact directly supported by the evidence.

Focus on the next missing relationship needed to answer the question.

If the question names alternatives, use those exact names.

Do not repeat previous reasoning.
Do not guess.
"""
                )



            raw_candidates = (
                sample_reasoning_candidates(
                    prompt,
                    K_SAMPLES,
                )
            )


            (
                candidates,
                pre_rejections,
            ) = filter_candidates(
                raw_candidates,
                steps,
            )





            total_repeated += sum(
                rejection["reason"]
                in {
                    "duplicate_within_batch",
                    "repeated_reasoning",
                }
                for rejection in pre_rejections
            )


            total_meta += sum(
                rejection["reason"]
                == "meta_reasoning"
                for rejection in pre_rejections
            )


            total_premature_final += sum(
                rejection["reason"]
                == "premature_final_answer"
                for rejection in pre_rejections
            )


            reports = [

                verify_reasoning_step(
                    candidate,
                    ex,
                    question,
                    steps,
                )

                for candidate in candidates
            ]


            total_contradicted += sum(
                report[
                    "confirmed_contradiction"
                ]
                for report in reports
            )


            total_conflicted += sum(
                report["conflicted"]
                for report in reports
            )


            total_unclear += sum(
                report["label"] == "unclear"
                for report in reports
            )




            selected = select_next_step(
                reports
            )


            trajectory_reports.append({

                "step_index":
                    step_index,

                "attempt":
                    attempt,

                "prompt_evidence":
                    prompt_evidence,

                "raw_candidates":
                    raw_candidates,

                "pre_rejections":
                    pre_rejections,

                "reports":
                    reports,

                "selected":
                    (
                        selected["text"]
                        if selected
                        else None
                    ),

                "selected_label":
                    (
                        selected["label"]
                        if selected
                        else None
                    ),

                "selected_grounded_unclear":
                    (
                        selected[
                            "grounded_unclear"
                        ]
                        if selected
                        else False
                    ),
            })




            if selected is not None:

                if selected[
                    "grounded_unclear"
                ]:

                    total_grounded_unclear_selected += 1


                if attempt > 0:
                    successful_repairs += 1


                break


            if attempt < MAX_RETRIES:

                total_resamples += 1

                continue



            return {

                "kind":
                    "stop",

                "answer":
                    "",

                "ans":
                    "",

                "reason":
                    "no_safe_grounded_continuation_after_retry",

                "steps":
                    steps,

                "trajectory_reports":
                    trajectory_reports,

                "answer_checks":
                    answer_checks,

                "num_contradicted_rejected":
                    total_contradicted,

                "num_conflicted_rejected":
                    total_conflicted,

                "num_unclear_seen":
                    total_unclear,

                "num_grounded_unclear_selected":
                    total_grounded_unclear_selected,

                "num_repetitions_rejected":
                    total_repeated,

                "num_meta_rejected":
                    total_meta,

                "num_premature_final_rejected":
                    total_premature_final,

                "num_resamples":
                    total_resamples,

                "num_successful_repairs":
                    successful_repairs,

                "answer_extraction_attempts":
                    answer_extraction_attempts,

                "answer_verification_failures":
                    answer_verification_failures,

                "claim_verification_attempts":
                    claim_verification_attempts,
            }



        steps.append(
            selected
        )


        answer_extraction_attempts += 1


        answer = (
            propose_direct_answer_full_claim(
                question,
                steps,
            )
        )


        if answer is None:

            answer_checks.append({

                "step_index":
                    step_index,

                "answer":
                    None,

                "passed":
                    False,

                "reason":
                    "answer_proposal_failure",
            })


            answer_verification_failures += 1

            continue


        verification = (
            verify_extracted_answer_full_claim(
                question,
                answer,
                steps,
            )
        )


        if verification.get(
            "claim"
        ) is not None:

            claim_verification_attempts += 1


        answer_checks.append({

            "step_index":
                step_index,

            "answer":
                answer,

            **verification,
        })




        if verification[
            "passed"
        ]:

            return {

                "kind":
                    "commit",

                "answer":
                    answer,

                "ans":
                    answer,

                "reason":
                    "answer_claim_verified",

                "final_claim":
                    verification.get(
                        "claim"
                    ),

                "steps":
                    steps,

                "trajectory_reports":
                    trajectory_reports,

                "answer_checks":
                    answer_checks,

                "num_contradicted_rejected":
                    total_contradicted,

                "num_conflicted_rejected":
                    total_conflicted,

                "num_unclear_seen":
                    total_unclear,

                "num_grounded_unclear_selected":
                    total_grounded_unclear_selected,

                "num_repetitions_rejected":
                    total_repeated,

                "num_meta_rejected":
                    total_meta,

                "num_premature_final_rejected":
                    total_premature_final,

                "num_resamples":
                    total_resamples,

                "num_successful_repairs":
                    successful_repairs,

                "answer_extraction_attempts":
                    answer_extraction_attempts,

                "answer_verification_failures":
                    answer_verification_failures,

                "claim_verification_attempts":
                    claim_verification_attempts,
            }


        answer_verification_failures += 1


    return {

        "kind":
            "stop",

        "answer":
            "",

        "ans":
            "",

        "reason":
            "max_steps_without_verified_answer_claim",

        "steps":
            steps,

        "trajectory_reports":
            trajectory_reports,

        "answer_checks":
            answer_checks,

        "num_contradicted_rejected":
            total_contradicted,

        "num_conflicted_rejected":
            total_conflicted,

        "num_unclear_seen":
            total_unclear,

        "num_grounded_unclear_selected":
            total_grounded_unclear_selected,

        "num_repetitions_rejected":
            total_repeated,

        "num_meta_rejected":
            total_meta,

        "num_premature_final_rejected":
            total_premature_final,

        "num_resamples":
            total_resamples,

        "num_successful_repairs":
            successful_repairs,

        "answer_extraction_attempts":
            answer_extraction_attempts,

        "answer_verification_failures":
            answer_verification_failures,

        "claim_verification_attempts":
            claim_verification_attempts,
    }


def normalize_gold_answer(s):

    def remove_articles(text):

        return re.sub(
            r"\b(a|an|the)\b",
            " ",
            text,
        )


    def white_space_fix(text):

        return " ".join(
            text.split()
        )


    def remove_punc(text):

        exclude = set(
            string.punctuation
        )

        return "".join(
            ch
            for ch in text
            if ch not in exclude
        )


    def lower(text):

        return text.lower()


    return white_space_fix(
        remove_articles(
            remove_punc(
                lower(
                    str(s)
                )
            )
        )
    )


def answer_exact_match(
    prediction,
    gold
):

    return float(
        normalize_gold_answer(
            prediction
        )
        ==
        normalize_gold_answer(
            gold
        )
    )


def answer_f1(
    prediction,
    gold
):

    pred_tokens = (
        normalize_gold_answer(
            prediction
        ).split()
    )


    gold_tokens = (
        normalize_gold_answer(
            gold
        ).split()
    )


    if len(pred_tokens) == 0:

        return float(
            len(gold_tokens) == 0
        )


    if len(gold_tokens) == 0:

        return 0.0


    common = Counter(
        pred_tokens
    ) & Counter(
        gold_tokens
    )


    num_same = sum(
        common.values()
    )


    if num_same == 0:
        return 0.0


    precision = (
        num_same
        /
        len(pred_tokens)
    )


    recall = (
        num_same
        /
        len(gold_tokens)
    )


    return (
        2
        *
        precision
        *
        recall
        /
        (
            precision
            +
            recall
        )
    )


full_pool = json.load(
    open(
        f"{DATA_DIR}/hedge_pre_rl_1000_full.json"
    )
)


q_by_qi = {

    int(record["question_index"]):
        record["question"]

    for record in full_pool
}


gold_by_qi = {

    int(record["question_index"]):
        str(
            record.get(
                "gold_answer",
                ""
            )
        )

    for record in full_pool
}


test_ids = set(

    int(record["question_index"])

    for record in json.load(
        open(
            f"{DATA_DIR}/dpo_test_questions.json"
        )
    )
)


eligible_non_test = [

    int(record["question_index"])

    for record in full_pool

    if (
        int(record["question_index"])
        not in test_ids
    )
]


validation_qi = (
    eligible_non_test[
        VAL_START:
        VAL_END
    ]
)


assert len(
    validation_qi
) == VALIDATION_N


pilot_qi = set(
    eligible_non_test[
        :PILOT_N
    ]
)


assert not (
    pilot_qi
    &
    set(validation_qi)
)


assert not (
    test_ids
    &
    set(validation_qi)
)


print()
print("Split verification:")
print(
    "  pilot overlap:",
    len(
        pilot_qi
        &
        set(validation_qi)
    )
)

print(
    "  test overlap:",
    len(
        test_ids
        &
        set(validation_qi)
    )
)

print(
    "  validation questions:",
    len(validation_qi)
)



# Reuse existing dataset if available.
if "dataset" not in globals():

    print(
        "\nLoading HotpotQA validation..."
    )

    dataset = load_dataset(
        "hotpotqa/hotpot_qa",
        "distractor",
        split="validation",
    )


question_to_example = {

    ex["question"]:
        ex

    for ex in dataset
}



records = []


if os.path.exists(
    CHECKPOINT_OUT
):

    print(
        "\nExisting checkpoint found."
    )


    with open(
        CHECKPOINT_OUT
    ) as f:

        checkpoint = json.load(f)


    old_ids = checkpoint.get(
        "validation_qi",
        []
    )


    # Only resume if the exact split matches.
    if old_ids == validation_qi:

        records = checkpoint.get(
            "records",
            []
        )


        print(
            f"Resuming from "
            f"{len(records)}/{VALIDATION_N}"
        )


    else:

        print(
            "Checkpoint belongs to a different split."
        )

        print(
            "Starting a fresh validation run."
        )

        records = []


done_ids = {

    int(record["question_index"])

    for record in records
}



print()
print("=" * 80)
print("STARTING 150-QUESTION HELD-OUT VALIDATION")
print("=" * 80)


for position, qi in enumerate(
    validation_qi,
    1
):

    if qi in done_ids:
        continue


    question = q_by_qi[
        qi
    ]


    gold = gold_by_qi[
        qi
    ]


    ex = question_to_example.get(
        question
    )


    print()
    print(
        f"[{position:03d}/{VALIDATION_N}] "
        f"qi={qi}"
    )


    print(
        "Q:",
        question[:130]
    )



    if ex is None:

        print(
            "  ERROR: Hotpot example not found."
        )


        record = {

            "question_index":
                qi,

            "question":
                question,

            "gold_answer":
                gold,

            "output": {
                "kind":
                    "error",

                "answer":
                    "",

                "reason":
                    "hotpot_example_not_found",
            },

            "offline_eval": {
                "answered":
                    False,

                "exact_match":
                    0.0,

                "f1":
                    0.0,
            },
        }


        records.append(
            record
        )


    else:



        result = run_question_full_claim(
            ex,
            question,
        )


        prediction = (
            result.get(
                "answer",
                ""
            )
            if result["kind"] == "commit"
            else ""
        )


        answered = (
            result["kind"]
            ==
            "commit"
        )


        em = (
            answer_exact_match(
                prediction,
                gold,
            )
            if answered
            else 0.0
        )


        f1 = (
            answer_f1(
                prediction,
                gold,
            )
            if answered
            else 0.0
        )


        print(
            "  outcome:",
            result["kind"]
        )


        print(
            "  reason:",
            result["reason"]
        )


        if answered:

            print(
                "  prediction:",
                prediction
            )

            print(
                "  gold:",
                gold
            )

            print(
                f"  EM={em:.0f} "
                f"F1={f1:.3f}"
            )

        else:

            print(
                "  ABSTAIN"
            )

            print(
                "  gold:",
                gold
            )


        record = {

            "question_index":
                qi,

            "question":
                question,

            # Gold stored only in output record,
            # after inference has completed.
            "gold_answer":
                gold,

            "output":
                result,

            "offline_eval": {

                "answered":
                    answered,

                "exact_match":
                    em,

                "f1":
                    f1,
            },
        }


        records.append(
            record
        )


    checkpoint_payload = {

        "architecture":
            "frozen_full_question_claim",

        "pilot_excluded":
            PILOT_N,

        "validation_n":
            VALIDATION_N,

        "validation_qi":
            validation_qi,

        "records":
            records,
    }


    with open(
        CHECKPOINT_OUT,
        "w"
    ) as f:

        json.dump(
            checkpoint_payload,
            f,
            indent=2,
        )


N = len(records)


valid_records = [

    record

    for record in records

    if record[
        "output"
    ][
        "kind"
    ]
    != "error"
]


N_VALID = len(
    valid_records
)


committed = [

    record

    for record
    in valid_records

    if (
        record[
            "output"
        ][
            "kind"
        ]
        ==
        "commit"
    )
]


abstained = [

    record

    for record
    in valid_records

    if (
        record[
            "output"
        ][
            "kind"
        ]
        ==
        "stop"
    )
]


n_commit = len(
    committed
)


n_abstain = len(
    abstained
)




n_exact = int(
    sum(
        record[
            "offline_eval"
        ][
            "exact_match"
        ]

        for record
        in valid_records
    )
)


n_answered_exact = int(
    sum(
        record[
            "offline_eval"
        ][
            "exact_match"
        ]

        for record
        in committed
    )
)



overall_mean_f1 = (
    np.mean(
        [
            record[
                "offline_eval"
            ][
                "f1"
            ]

            for record
            in valid_records
        ]
    )
    if valid_records
    else 0.0
)


selective_mean_f1 = (
    np.mean(
        [
            record[
                "offline_eval"
            ][
                "f1"
            ]

            for record
            in committed
        ]
    )
    if committed
    else 0.0
)



coverage = (
    n_commit
    /
    N_VALID
    if N_VALID
    else 0.0
)


selective_em = (
    n_answered_exact
    /
    n_commit
    if n_commit
    else 0.0
)


overall_em = (
    n_exact
    /
    N_VALID
    if N_VALID
    else 0.0
)


wrong_answered_em = (
    n_commit
    -
    n_answered_exact
)


wrong_answer_rate_total = (
    wrong_answered_em
    /
    N_VALID
    if N_VALID
    else 0.0
)


wrong_answer_rate_answered = (
    wrong_answered_em
    /
    n_commit
    if n_commit
    else 0.0
)




COUNTER_FIELDS = [

    "num_contradicted_rejected",

    "num_conflicted_rejected",

    "num_unclear_seen",

    "num_grounded_unclear_selected",

    "num_repetitions_rejected",

    "num_meta_rejected",

    "num_premature_final_rejected",

    "num_resamples",

    "num_successful_repairs",

    "answer_extraction_attempts",

    "answer_verification_failures",

    "claim_verification_attempts",
]


interventions = {}


for field in COUNTER_FIELDS:

    interventions[field] = int(
        sum(
            record[
                "output"
            ].get(
                field,
                0
            )

            for record
            in valid_records
        )
    )



stop_reasons = Counter(

    record[
        "output"
    ][
        "reason"
    ]

    for record
    in abstained
)



print()
print("=" * 80)
print("FINAL 150-QUESTION VALIDATION RESULTS")
print("=" * 80)


print(
    f"Questions evaluated:         "
    f"{N_VALID}"
)


print(
    f"Committed:                   "
    f"{n_commit}"
)


print(
    f"Abstained:                   "
    f"{n_abstain}"
)


print(
    f"Coverage:                    "
    f"{coverage:.3f}"
)


print()
print("--- Exact Match ---")


print(
    f"Correct / all:               "
    f"{n_exact}/{N_VALID}"
)


print(
    f"Overall EM accuracy:         "
    f"{overall_em:.3f}"
)


print(
    f"Selective EM accuracy:       "
    f"{selective_em:.3f}"
)


print(
    f"Wrong committed answers:     "
    f"{wrong_answered_em}"
)


print(
    f"Wrong-answer rate / total:   "
    f"{wrong_answer_rate_total:.3f}"
)


print(
    f"Wrong-answer rate / answered:"
    f" {wrong_answer_rate_answered:.3f}"
)


print()
print("--- Hotpot Token F1 ---")


print(
    f"Overall mean F1:             "
    f"{overall_mean_f1:.3f}"
)


print(
    f"Selective mean F1:           "
    f"{selective_mean_f1:.3f}"
)


print()
print("--- Stop Reasons ---")


for reason, count in (
    stop_reasons.items()
):

    print(
        f"{reason}: {count}"
    )


print()
print("--- Verifier Interventions ---")


for field, value in (
    interventions.items()
):

    print(
        f"{field}: {value}"
    )



final_payload = {

    "architecture":
        "frozen_full_question_claim",

    "split": {

        "pilot_excluded":
            PILOT_N,

        "heldout_validation_n":
            VALIDATION_N,

        "validation_slice":
            [
                VAL_START,
                VAL_END,
            ],

        "validation_qi":
            validation_qi,

        "test_overlap":
            len(
                set(validation_qi)
                &
                test_ids
            ),

        "pilot_overlap":
            len(
                set(validation_qi)
                &
                pilot_qi
            ),
    },


    "config": {

        "K_SAMPLES":
            K_SAMPLES,

        "MAX_STEPS":
            MAX_STEPS,

        "MAX_RETRIES":
            MAX_RETRIES,

        "STEP_RETRIEVE_K":
            STEP_RETRIEVE_K,

        "QUESTION_RETRIEVE_K":
            QUESTION_RETRIEVE_K,

        "TRAJECTORY_RETRIEVE_K":
            TRAJECTORY_RETRIEVE_K,

        "SUPPORT_THR":
            SUPPORT_THR,

        "CONTRA_THR":
            CONTRA_THR,

        "CONTRA_ADVANTAGE":
            CONTRA_ADVANTAGE,

        "CONFLICT_THR":
            CONFLICT_THR,

        "UNCLEAR_CON_MAX":
            UNCLEAR_CON_MAX,

        "UNCLEAR_EVIDENCE_OVERLAP_MIN":
            UNCLEAR_EVIDENCE_OVERLAP_MIN,

        "UNCLEAR_FOCUS_MIN":
            UNCLEAR_FOCUS_MIN,

        "CLAIM_SUPPORT_THR":
            CLAIM_SUPPORT_THR,

        "CLAIM_CONTRA_THR":
            CLAIM_CONTRA_THR,

        "CLAIM_CONTRA_ADVANTAGE":
            CLAIM_CONTRA_ADVANTAGE,

        "CLAIM_CONFLICT_THR":
            CLAIM_CONFLICT_THR,
    },


    "metrics": {

        "n":
            N_VALID,

        "committed":
            n_commit,

        "abstained":
            n_abstain,

        "coverage":
            coverage,

        "exact_correct":
            n_exact,

        "overall_exact_match":
            overall_em,

        "selective_exact_match":
            selective_em,

        "wrong_committed_answers":
            wrong_answered_em,

        "wrong_answer_rate_total":
            wrong_answer_rate_total,

        "wrong_answer_rate_answered":
            wrong_answer_rate_answered,

        "overall_mean_f1":
            float(
                overall_mean_f1
            ),

        "selective_mean_f1":
            float(
                selective_mean_f1
            ),

        "stop_reasons":
            dict(
                stop_reasons
            ),

        "interventions":
            interventions,
    },


    "records":
        records,
}


with open(
    VALIDATION_OUT,
    "w"
) as f:

    json.dump(
        final_payload,
        f,
        indent=2,
    )


print()
print(
    "Saved final validation:"
)

print(
    VALIDATION_OUT
)


print()
print(
    "Checkpoint:"
)

print(
    CHECKPOINT_OUT
)

FROZEN FULL-CLAIM VALIDATION
Pilot questions excluded : 25
Held-out validation size : 150


Split verification:
  pilot overlap: 0
  test overlap: 0
  validation questions: 150

STARTING 150-QUESTION HELD-OUT VALIDATION

[001/150] qi=31
Q: In the NASA mission where Moon trees were taken into space, what was the nickname of the Command Module?
  outcome: stop
  reason: no_safe_grounded_continuation_after_retry
  ABSTAIN
  gold: "Kitty Hawk"

[002/150] qi=32
Q: Which comic series involves characters such as Nick Fury and Baron von Strucker?
  outcome: commit
  reason: answer_claim_verified
  prediction: Nick Fury, Agent of S.H.I.E.L.D
  gold: Marvel
  EM=0 F1=0.000

[003/150] qi=35
Q: New York State Route 9R rejoins its parent in a hamlet located  in what New York County?
  outcome: commit
  reason: answer_claim_verified
  prediction: Albany County
  gold: Albany
  EM=0 F1=0.667

[004/150] qi=36
Q: 12 Years a Slave starred what British actor born 10 July 1977)
  outcome: commit
  reason:

In [ ]:
import os
import json
from openai import OpenAI


DATA_DIR = "/content/drive/MyDrive/hedge_run"

VERIFIER_JSON = (
    f"{DATA_DIR}/verifier_full_claim_validation_150.json"
)

OUT_JSON = (
    f"{DATA_DIR}/verifier_full_claim_validation_150_judged.json"
)

JUDGE_CACHE = (
    f"{DATA_DIR}/verifier_full_claim_validation_150_judge_cache.json"
)

# Existing baseline judgments, for direct same-split comparison
BASE_JUDGE_JSON = (
    f"{DATA_DIR}/rejudged_1000.json"
)


try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

oai = OpenAI(
    api_key=OPENAI_API_KEY
)



def judge(q, gold, ans):

    if not ans or not ans.strip():
        return False, ""


    prompt = (
        f"Question: {q}\n"
        f"Gold answer: {gold}\n"
        f"Predicted: {ans}\n"
        "Is the predicted answer correct? "
        "Accept paraphrases. Reply yes or no."
    )


    r = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        max_tokens=4,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )


    raw = (
        r.choices[0]
        .message
        .content
        .strip()
    )


    correct = (
        raw.lower()
        .startswith("y")
    )


    return correct, raw




with open(
    VERIFIER_JSON
) as f:

    data = json.load(f)


records = data["records"]


print(
    "records:",
    len(records)
)


if os.path.exists(
    JUDGE_CACHE
):

    with open(
        JUDGE_CACHE
    ) as f:

        cache = json.load(f)

    print(
        "loaded cached judgments:",
        len(cache)
    )

else:

    cache = {}



judged_records = []


for i, rec in enumerate(
    records,
    1
):

    qi = int(
        rec["question_index"]
    )


    question = rec[
        "question"
    ]


    gold = rec[
        "gold_answer"
    ]


    output = rec[
        "output"
    ]


    kind = output[
        "kind"
    ]


    if kind != "commit":

        judged = {

            "question_index":
                qi,

            "question":
                question,

            "gold_answer":
                gold,

            "prediction":
                "",

            "kind":
                "abstain",

            "judge_correct":
                None,

            "judge_raw":
                None,
        }


        judged_records.append(
            judged
        )


        print(
            f"[{i:03d}/{len(records)}] "
            f"qi={qi} ABSTAIN"
        )

        continue



    prediction = str(
        output.get(
            "answer",
            ""
        )
    ).strip()


    key = str(qi)




    cached = cache.get(
        key
    )


    if (
        cached is not None

        and

        cached.get(
            "prediction"
        )
        ==
        prediction

        and

        cached.get(
            "gold_answer"
        )
        ==
        gold
    ):

        correct = bool(
            cached[
                "judge_correct"
            ]
        )


        raw = cached.get(
            "judge_raw",
            ""
        )


        source = "cached"


    else:


        correct, raw = judge(
            question,
            gold,
            prediction,
        )


        source = "new"



        cache[
            key
        ] = {

            "question_index":
                qi,

            "question":
                question,

            "gold_answer":
                gold,

            "prediction":
                prediction,

            "judge_correct":
                bool(correct),

            "judge_raw":
                raw,
        }


        with open(
            JUDGE_CACHE,
            "w"
        ) as f:

            json.dump(
                cache,
                f,
                indent=2,
                ensure_ascii=False,
            )


    judged = {

        "question_index":
            qi,

        "question":
            question,

        "gold_answer":
            gold,

        "prediction":
            prediction,

        "kind":
            (
                "commit_correct"
                if correct
                else "commit_wrong"
            ),

        "judge_correct":
            bool(correct),

        "judge_raw":
            raw,
    }


    judged_records.append(
        judged
    )


    symbol = (
        "✓"
        if correct
        else "✗"
    )


    print(
        f"[{i:03d}/{len(records)}] "
        f"qi={qi} "
        f"{symbol} "
        f"{prediction[:45]} "
        f"({source})"
    )




N = len(
    judged_records
)


committed = [

    r

    for r in judged_records

    if r["kind"]
    in {
        "commit_correct",
        "commit_wrong",
    }
]


abstained = [

    r

    for r in judged_records

    if r["kind"]
    ==
    "abstain"
]


correct_commits = [

    r

    for r in committed

    if r[
        "judge_correct"
    ]
]


wrong_commits = [

    r

    for r in committed

    if not r[
        "judge_correct"
    ]
]


n_commit = len(
    committed
)

n_abstain = len(
    abstained
)

n_correct = len(
    correct_commits
)

n_wrong = len(
    wrong_commits
)


coverage = (
    n_commit
    /
    N
)


selective_accuracy = (
    n_correct
    /
    n_commit

    if n_commit
    else 0.0
)


overall_accuracy = (
    n_correct
    /
    N
)


wrong_rate_total = (
    n_wrong
    /
    N
)


wrong_rate_answered = (
    n_wrong
    /
    n_commit

    if n_commit
    else 0.0
)


validation_ids = {

    int(
        r[
            "question_index"
        ]
    )

    for r in judged_records
}


baseline_correct = {}


if os.path.exists(
    BASE_JUDGE_JSON
):

    with open(
        BASE_JUDGE_JSON
    ) as f:

        baseline_raw = json.load(f)


    baseline_correct = {

        int(r["question_index"]):
            bool(
                r["judge_correct"]
            )

        for r in baseline_raw

        if (
            int(
                r["question_index"]
            )
            in validation_ids
        )
    }



good_catches = 0

over_abstentions = 0

kept_baseline_correct = 0

kept_baseline_wrong = 0


if baseline_correct:

    for r in judged_records:

        qi = int(
            r["question_index"]
        )


        if qi not in baseline_correct:
            continue


        base_ok = baseline_correct[
            qi
        ]


        if r["kind"] == "abstain":

            if base_ok:
                over_abstentions += 1
            else:
                good_catches += 1


        else:

            if base_ok:
                kept_baseline_correct += 1
            else:
                kept_baseline_wrong += 1


    n_base = len(
        baseline_correct
    )


    n_base_correct = sum(
        baseline_correct.values()
    )


    n_base_wrong = (
        n_base
        -
        n_base_correct
    )


    baseline_accuracy = (
        n_base_correct
        /
        n_base
    )


else:

    n_base = 0
    n_base_correct = 0
    n_base_wrong = 0
    baseline_accuracy = None



print()
print("=" * 80)
print("SEMANTIC JUDGE RESULTS")
print("=" * 80)


print(
    f"Total questions:                    "
    f"{N}"
)


print(
    f"Committed:                          "
    f"{n_commit}"
)


print(
    f"Abstained:                          "
    f"{n_abstain}"
)


print(
    f"Coverage:                           "
    f"{coverage:.3f}"
)


print()
print("--- Semantic correctness ---")


print(
    f"Correct committed answers:          "
    f"{n_correct}"
)


print(
    f"Wrong committed answers:            "
    f"{n_wrong}"
)


print(
    f"Selective semantic accuracy:        "
    f"{selective_accuracy:.3f}"
)


print(
    f"Overall semantic accuracy:          "
    f"{overall_accuracy:.3f}"
)


print(
    f"Wrong-answer rate / total:          "
    f"{wrong_rate_total:.3f}"
)


print(
    f"Wrong-answer rate / answered:       "
    f"{wrong_rate_answered:.3f}"
)



if baseline_correct:

    print()
    print("--- BASELINE ON EXACT SAME 150 QUESTIONS ---")


    print(
        f"Baseline records found:              "
        f"{n_base}"
    )


    print(
        f"Baseline correct:                    "
        f"{n_base_correct}"
    )


    print(
        f"Baseline wrong:                      "
        f"{n_base_wrong}"
    )


    print(
        f"Baseline semantic accuracy:          "
        f"{baseline_accuracy:.3f}"
    )


    print()
    print("--- Abstention behaviour ---")


    print(
        f"Good catches "
        f"(baseline wrong -> abstain):     "
        f"{good_catches}"
    )


    print(
        f"Over-abstentions "
        f"(baseline right -> abstain):    "
        f"{over_abstentions}"
    )


    if n_base_wrong:

        abstention_recall = (
            good_catches
            /
            n_base_wrong
        )


        print(
            f"Abstention recall on baseline wrong: "
            f"{abstention_recall:.3f}"
        )


    if n_base_correct:

        over_abstention_rate = (
            over_abstentions
            /
            n_base_correct
        )


        print(
            f"Over-abstention rate on base correct:"
            f" {over_abstention_rate:.3f}"
        )


    print()
    print("--- Accuracy gain among returned answers ---")


    print(
        f"Baseline accuracy:                   "
        f"{baseline_accuracy:.3f}"
    )


    print(
        f"Verifier selective accuracy:         "
        f"{selective_accuracy:.3f}"
    )


    print(
        f"Absolute gain:                       "
        f"{selective_accuracy - baseline_accuracy:+.3f}"
    )



result = {

    "judge": {

        "model":
            "gpt-4o-mini",

        "temperature":
            0,

        "max_tokens":
            4,

        "prompt":
            (
                "Question: {q}\\n"
                "Gold answer: {gold}\\n"
                "Predicted: {ans}\\n"
                "Is the predicted answer correct? "
                "Accept paraphrases. Reply yes or no."
            ),
    },


    "metrics": {

        "n":
            N,

        "committed":
            n_commit,

        "abstained":
            n_abstain,

        "coverage":
            coverage,

        "correct_committed":
            n_correct,

        "wrong_committed":
            n_wrong,

        "selective_semantic_accuracy":
            selective_accuracy,

        "overall_semantic_accuracy":
            overall_accuracy,

        "wrong_answer_rate_total":
            wrong_rate_total,

        "wrong_answer_rate_answered":
            wrong_rate_answered,
    },


    "baseline_same_split": {

        "n":
            n_base,

        "correct":
            n_base_correct,

        "wrong":
            n_base_wrong,

        "accuracy":
            baseline_accuracy,

        "good_catches":
            good_catches,

        "over_abstentions":
            over_abstentions,
    },


    "records":
        judged_records,
}


with open(
    OUT_JSON,
    "w"
) as f:

    json.dump(
        result,
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print(
    "Saved:"
)

print(
    OUT_JSON
)



print()
print("=" * 80)
print("SEMANTICALLY WRONG COMMITTED ANSWERS")
print("=" * 80)


for r in wrong_commits:

    print()
    print(
        f"qi={r['question_index']}"
    )

    print(
        "Q:",
        r["question"]
    )

    print(
        "Prediction:",
        r["prediction"]
    )

    print(
        "Gold:",
        r["gold_answer"]
    )

records: 150
loaded cached judgments: 113
[001/150] qi=31 ABSTAIN
[002/150] qi=32 ✓ Nick Fury, Agent of S.H.I.E.L.D (cached)
[003/150] qi=35 ✓ Albany County (cached)
[004/150] qi=36 ✓ Chiwetel Ejiofor (cached)
[005/150] qi=37 ✓ Agra (cached)
[006/150] qi=39 ✓ Mwabvi river (cached)
[007/150] qi=40 ✓ Smoothie King Center (cached)
[008/150] qi=41 ✓ segues (cached)
[009/150] qi=42 ✗ Lady Charlotte Elliot (cached)
[010/150] qi=43 ✓ World War II (cached)
[011/150] qi=44 ✓ Oklahoma State (cached)
[012/150] qi=45 ✓ Brady Seals (cached)
[013/150] qi=46 ✓ Nusretiye Mosque (cached)
[014/150] qi=47 ✗ University of Colorado (cached)
[015/150] qi=48 ✓ Turkey (cached)
[016/150] qi=51 ✓ No (cached)
[017/150] qi=52 ABSTAIN
[018/150] qi=54 ✓ Netherlands (cached)
[019/150] qi=55 ✓ 1950 (cached)
[020/150] qi=57 ABSTAIN
[021/150] qi=58 ✓ Dirk Nowitzki (cached)
[022/150] qi=59 ✓ Henry Lau (cached)
[023/150] qi=60 ABSTAIN
[024/150] qi=61 ✓ Kalokuokamaile (cached)
[025/150] qi=62 ✓ 12 (cached)
[026/150] qi=63

In [ ]:


import os
import re
import json
from collections import Counter

from openai import OpenAI


required = [
    "run_question_full_claim",
    "dataset",
    "DATA_DIR",
]

missing = [
    name
    for name in required
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Frozen verifier is not loaded in this runtime. "
        f"Missing: {missing}\n"
        "Run the frozen full-claim verifier/validation definition cell first."
    )


TEST_JSON = (
    f"{DATA_DIR}/dpo_test_questions.json"
)

FULL_JSON = (
    f"{DATA_DIR}/hedge_pre_rl_1000_full.json"
)

BASE_JUDGE_JSON = (
    f"{DATA_DIR}/rejudged_1000.json"
)

TEST_PROGRESS = (
    f"{DATA_DIR}/verifier_full_claim_FINAL_TEST_progress.json"
)

TEST_RAW_OUT = (
    f"{DATA_DIR}/verifier_full_claim_FINAL_TEST_raw.json"
)

JUDGE_CACHE = (
    f"{DATA_DIR}/verifier_full_claim_FINAL_TEST_judge_cache.json"
)

FINAL_OUT = (
    f"{DATA_DIR}/verifier_full_claim_FINAL_TEST_judged.json"
)



try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY not found."
    )


oai = OpenAI(
    api_key=OPENAI_API_KEY
)




with open(FULL_JSON) as f:
    full_pool = json.load(f)


full_by_qi = {
    int(r["question_index"]): r
    for r in full_pool
}


with open(TEST_JSON) as f:
    test_manifest = json.load(f)


test_qi = [
    int(r["question_index"])
    for r in test_manifest
]


if len(test_qi) != len(set(test_qi)):
    raise RuntimeError(
        "Duplicate question_index values in test split."
    )




q_to_ex = {
    ex["question"]: ex
    for ex in dataset
}


test_items = []


for qi in test_qi:

    row = full_by_qi.get(qi)

    if row is None:
        raise KeyError(
            f"qi={qi} missing from {FULL_JSON}"
        )


    question = row["question"]

    ex = q_to_ex.get(question)


    if ex is None:
        raise KeyError(
            f"HotpotQA example not found for qi={qi}\n"
            f"Question: {question}"
        )


    test_items.append({
        "question_index": qi,
        "question": question,

        # Gold is stored for OFFLINE evaluation only.
        # It will never be passed to run_question_full_claim().
        "gold_answer": ex["answer"],

        "example": ex,
    })


N_TEST = len(test_items)


print("=" * 80)
print("FINAL FROZEN TEST")
print("=" * 80)

print(
    "Test questions:",
    N_TEST
)

print(
    "Unique IDs:",
    len(set(test_qi))
)


validation_path = (
    f"{DATA_DIR}/verifier_full_claim_validation_150.json"
)


if os.path.exists(validation_path):

    with open(validation_path) as f:
        validation_data = json.load(f)


    validation_ids = set(
        int(x)
        for x in validation_data[
            "split"
        ][
            "validation_qi"
        ]
    )


    overlap = (
        validation_ids
        &
        set(test_qi)
    )


    print(
        "Validation/test overlap:",
        len(overlap)
    )


    if overlap:
        raise RuntimeError(
            f"TEST LEAKAGE: {len(overlap)} "
            "test IDs appeared in validation."
        )


print()



if os.path.exists(TEST_PROGRESS):

    with open(TEST_PROGRESS) as f:
        progress_payload = json.load(f)


    if progress_payload.get(
        "test_qi"
    ) == test_qi:

        test_records = progress_payload.get(
            "records",
            []
        )


        print(
            f"Resuming inference from "
            f"{len(test_records)}/{N_TEST}"
        )


    else:

        raise RuntimeError(
            "Existing test checkpoint uses a different test split. "
            "Rename/delete the checkpoint before continuing."
        )

else:

    test_records = []


done_qi = {
    int(r["question_index"])
    for r in test_records
}


print()
print("=" * 80)
print("RUNNING FROZEN VERIFIER ON TEST SET")
print("=" * 80)


for position, item in enumerate(
    test_items,
    start=1,
):

    qi = item[
        "question_index"
    ]


    if qi in done_qi:
        continue


    question = item[
        "question"
    ]


    ex = item[
        "example"
    ]


    print()
    print(
        f"[{position:03d}/{N_TEST}] "
        f"qi={qi}"
    )

    print(
        "Q:",
        question[:130]
    )



    try:

        output = run_question_full_claim(
            ex,
            question,
        )


    except Exception as exc:

        print(
            "  ERROR:",
            type(exc).__name__,
            str(exc),
        )


        output = {
            "kind": "error",
            "answer": "",
            "reason": (
                f"{type(exc).__name__}: "
                f"{str(exc)}"
            ),
        }


    prediction = (
        str(
            output.get(
                "answer",
                ""
            )
        ).strip()
    )


    print(
        "  outcome:",
        output.get(
            "kind"
        )
    )


    print(
        "  reason:",
        output.get(
            "reason"
        )
    )


    if output.get(
        "kind"
    ) == "commit":

        print(
            "  prediction:",
            prediction
        )

    else:

        print(
            "  ABSTAIN"
        )


    test_records.append({
        "question_index":
            qi,

        "question":
            question,

        "gold_answer":
            item["gold_answer"],

        "output":
            output,
    })



    with open(
        TEST_PROGRESS,
        "w",
    ) as f:

        json.dump(
            {
                "architecture":
                    "frozen_full_question_claim",

                "split":
                    "final_test",

                "test_qi":
                    test_qi,

                "records":
                    test_records,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )



with open(
    TEST_RAW_OUT,
    "w",
) as f:

    json.dump(
        {
            "architecture":
                "frozen_full_question_claim",

            "split":
                "final_test",

            "n":
                len(test_records),

            "test_qi":
                test_qi,

            "records":
                test_records,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print(
    "Raw test inference saved:"
)

print(
    TEST_RAW_OUT
)



def baseline_identical_judge(
    q,
    gold,
    ans,
):

    if not ans.strip():
        return False, ""


    p = (
        f"Question: {q}\n"
        f"Correct answer: {gold}\n"
        f"Model's answer: {ans}\n\n"
        "Is the model's answer correct? "
        "Accept paraphrases. "
        "Reply briefly then 'Verdict: yes' or 'Verdict: no'."
    )


    r = oai.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        max_tokens=120,
        messages=[
            {
                "role": "user",
                "content": p,
            }
        ],
    )


    raw = (
        r.choices[0]
        .message
        .content
        .strip()
    )


    m = re.search(
        r"verdict:\s*(yes|no)",
        raw.lower(),
    )


    correct = bool(
        m
        and
        m.group(1)
        ==
        "yes"
    )


    return correct, raw




if os.path.exists(
    JUDGE_CACHE
):

    with open(
        JUDGE_CACHE
    ) as f:

        judge_cache = json.load(f)


    print()
    print(
        "Loaded cached test judgments:",
        len(judge_cache)
    )

else:

    judge_cache = {}



print()
print("=" * 80)
print("SEMANTIC JUDGING — IDENTICAL TO BASELINES")
print("=" * 80)


judged_records = []


for i, rec in enumerate(
    test_records,
    1,
):

    qi = int(
        rec["question_index"]
    )


    q = rec[
        "question"
    ]


    gold = rec[
        "gold_answer"
    ]


    output = rec[
        "output"
    ]


    kind = output.get(
        "kind"
    )



    if kind != "commit":

        judged_records.append({
            "question_index":
                qi,

            "question":
                q,

            "gold_answer":
                gold,

            "prediction":
                "",

            "kind":
                (
                    "error"
                    if kind == "error"
                    else "abstain"
                ),

            "judge_correct":
                None,

            "judge_raw":
                None,
        })


        print(
            f"[{i:03d}/{N_TEST}] "
            f"qi={qi} "
            f"{'ERROR' if kind == 'error' else 'ABSTAIN'}"
        )


        continue



    prediction = str(
        output.get(
            "answer",
            ""
        )
    ).strip()


    key = str(qi)


    cached = judge_cache.get(
        key
    )


    if (
        cached is not None
        and
        cached.get(
            "prediction"
        )
        ==
        prediction
        and
        cached.get(
            "gold_answer"
        )
        ==
        gold
    ):

        correct = bool(
            cached[
                "judge_correct"
            ]
        )

        raw = cached.get(
            "judge_raw",
            ""
        )

        source = "cached"


    else:

        correct, raw = (
            baseline_identical_judge(
                q,
                gold,
                prediction,
            )
        )


        judge_cache[key] = {
            "question_index":
                qi,

            "question":
                q,

            "gold_answer":
                gold,

            "prediction":
                prediction,

            "judge_correct":
                bool(correct),

            "judge_raw":
                raw,
        }


        # Save after every API call
        with open(
            JUDGE_CACHE,
            "w",
        ) as f:

            json.dump(
                judge_cache,
                f,
                indent=2,
                ensure_ascii=False,
            )


        source = "new"


    judged_records.append({
        "question_index":
            qi,

        "question":
            q,

        "gold_answer":
            gold,

        "prediction":
            prediction,

        "kind":
            (
                "commit_correct"
                if correct
                else "commit_wrong"
            ),

        "judge_correct":
            bool(correct),

        "judge_raw":
            raw,
    })


    print(
        f"[{i:03d}/{N_TEST}] "
        f"qi={qi} "
        f"{'✓' if correct else '✗'} "
        f"{prediction[:55]} "
        f"({source})"
    )



valid_records = [
    r
    for r in judged_records
    if r["kind"] != "error"
]


committed = [
    r
    for r in valid_records
    if r["kind"]
    in {
        "commit_correct",
        "commit_wrong",
    }
]


correct_commits = [
    r
    for r in committed
    if r["judge_correct"]
]


wrong_commits = [
    r
    for r in committed
    if not r["judge_correct"]
]


abstained = [
    r
    for r in valid_records
    if r["kind"] == "abstain"
]


N = len(
    valid_records
)

n_commit = len(
    committed
)

n_correct = len(
    correct_commits
)

n_wrong = len(
    wrong_commits
)

n_abstain = len(
    abstained
)


coverage = (
    n_commit / N
    if N
    else 0.0
)


selective_accuracy = (
    n_correct / n_commit
    if n_commit
    else 0.0
)


overall_accuracy = (
    n_correct / N
    if N
    else 0.0
)


wrong_rate_total = (
    n_wrong / N
    if N
    else 0.0
)


wrong_rate_answered = (
    n_wrong / n_commit
    if n_commit
    else 0.0
)




with open(
    BASE_JUDGE_JSON
) as f:

    baseline_rows = json.load(f)


baseline_correct = {
    int(r["question_index"]):
        bool(r["judge_correct"])
    for r in baseline_rows
    if int(r["question_index"])
    in set(test_qi)
}


if len(
    baseline_correct
) != len(
    test_qi
):

    print()
    print(
        "WARNING:"
    )

    print(
        f"Found baseline judgments for "
        f"{len(baseline_correct)}/{len(test_qi)} "
        f"test questions."
    )


transition = Counter()


for r in valid_records:

    qi = int(
        r["question_index"]
    )


    if qi not in baseline_correct:
        continue


    base = (
        "base_correct"
        if baseline_correct[qi]
        else "base_wrong"
    )


    if r["kind"] == "abstain":

        verifier = "abstain"


    elif r["judge_correct"]:

        verifier = "verifier_correct"


    else:

        verifier = "verifier_wrong"


    transition[
        (base, verifier)
    ] += 1


n_base = len(
    baseline_correct
)


n_base_correct = sum(
    baseline_correct.values()
)


n_base_wrong = (
    n_base
    -
    n_base_correct
)


baseline_accuracy = (
    n_base_correct
    /
    n_base
    if n_base
    else 0.0
)



good_catches = transition[
    (
        "base_wrong",
        "abstain",
    )
]


over_abstentions = transition[
    (
        "base_correct",
        "abstain",
    )
]


repaired_errors = transition[
    (
        "base_wrong",
        "verifier_correct",
    )
]


residual_errors = transition[
    (
        "base_wrong",
        "verifier_wrong",
    )
]


retained_correct = transition[
    (
        "base_correct",
        "verifier_correct",
    )
]


introduced_errors = transition[
    (
        "base_correct",
        "verifier_wrong",
    )
]


abstention_recall = (
    good_catches
    /
    n_base_wrong
    if n_base_wrong
    else 0.0
)


over_abstention_rate = (
    over_abstentions
    /
    n_base_correct
    if n_base_correct
    else 0.0
)


repair_rate = (
    repaired_errors
    /
    n_base_wrong
    if n_base_wrong
    else 0.0
)



stop_reasons = Counter()


for raw_record in test_records:

    output = raw_record[
        "output"
    ]


    if output.get(
        "kind"
    ) == "stop":

        stop_reasons[
            output.get(
                "reason",
                "unknown"
            )
        ] += 1



COUNTER_FIELDS = [
    "num_contradicted_rejected",
    "num_conflicted_rejected",
    "num_unclear_seen",
    "num_grounded_unclear_selected",
    "num_repetitions_rejected",
    "num_meta_rejected",
    "num_premature_final_rejected",
    "num_resamples",
    "num_successful_repairs",
    "answer_extraction_attempts",
    "answer_verification_failures",
    "claim_verification_attempts",
]


interventions = {}


for field in COUNTER_FIELDS:

    interventions[field] = int(
        sum(
            r[
                "output"
            ].get(
                field,
                0,
            )
            for r in test_records
        )
    )



print()
print("=" * 80)
print("FINAL TEST RESULTS")
print("=" * 80)


print(
    f"Test questions:                      "
    f"{N}"
)


print(
    f"Committed:                           "
    f"{n_commit}"
)


print(
    f"Abstained:                           "
    f"{n_abstain}"
)


print(
    f"Coverage:                            "
    f"{coverage:.3f}"
)


print()
print("--- Semantic correctness ---")


print(
    f"Correct committed:                   "
    f"{n_correct}"
)


print(
    f"Wrong committed:                     "
    f"{n_wrong}"
)


print(
    f"Selective semantic accuracy:         "
    f"{selective_accuracy:.3f}"
)


print(
    f"Overall semantic accuracy:           "
    f"{overall_accuracy:.3f}"
)


print(
    f"Wrong-answer rate / total:           "
    f"{wrong_rate_total:.3f}"
)


print(
    f"Wrong-answer rate / answered:        "
    f"{wrong_rate_answered:.3f}"
)




print()
print("--- BASELINE ON SAME TEST ---")


print(
    f"Baseline records:                    "
    f"{n_base}"
)


print(
    f"Baseline correct:                    "
    f"{n_base_correct}"
)


print(
    f"Baseline wrong:                      "
    f"{n_base_wrong}"
)


print(
    f"Baseline accuracy:                   "
    f"{baseline_accuracy:.3f}"
)


print(
    f"Verifier selective accuracy:         "
    f"{selective_accuracy:.3f}"
)


print(
    f"Selective accuracy gain:             "
    f"{selective_accuracy - baseline_accuracy:+.3f}"
)



print()
print("--- BASELINE -> VERIFIER TRANSITIONS ---")


print(
    f"Base correct -> verifier correct:    "
    f"{retained_correct}"
)


print(
    f"Base correct -> verifier wrong:      "
    f"{introduced_errors}"
)


print(
    f"Base correct -> abstain:             "
    f"{over_abstentions}"
)


print()
print(
    f"Base wrong   -> verifier correct:    "
    f"{repaired_errors}"
)


print(
    f"Base wrong   -> verifier wrong:      "
    f"{residual_errors}"
)


print(
    f"Base wrong   -> abstain:             "
    f"{good_catches}"
)


print()
print("--- Reliability behaviour ---")


print(
    f"Abstention recall on base-wrong:     "
    f"{abstention_recall:.3f}"
)


print(
    f"Over-abstention on base-correct:     "
    f"{over_abstention_rate:.3f}"
)


print(
    f"Baseline-error repair rate:          "
    f"{repair_rate:.3f}"
)




print()
print("--- Stop reasons ---")


for reason, count in (
    stop_reasons.items()
):

    print(
        f"{reason}: {count}"
    )



final_payload = {

    "architecture":
        "frozen_full_question_claim",

    "split":
        "final_test",

    "test_qi":
        test_qi,

    "judge": {

        "model":
            "gpt-4o-mini",

        "temperature":
            0,

        "max_tokens":
            120,

        "prompt":
            (
                "Question: {q}\\n"
                "Correct answer: {gold}\\n"
                "Model's answer: {ans}\\n\\n"
                "Is the model's answer correct? "
                "Accept paraphrases. "
                "Reply briefly then "
                "'Verdict: yes' or 'Verdict: no'."
            ),
    },


    "metrics": {

        "n":
            N,

        "committed":
            n_commit,

        "abstained":
            n_abstain,

        "coverage":
            coverage,

        "correct_committed":
            n_correct,

        "wrong_committed":
            n_wrong,

        "selective_semantic_accuracy":
            selective_accuracy,

        "overall_semantic_accuracy":
            overall_accuracy,

        "wrong_answer_rate_total":
            wrong_rate_total,

        "wrong_answer_rate_answered":
            wrong_rate_answered,
    },


    "baseline_same_test": {

        "n":
            n_base,

        "correct":
            n_base_correct,

        "wrong":
            n_base_wrong,

        "accuracy":
            baseline_accuracy,
    },


    "transitions": {

        "base_correct_to_verifier_correct":
            retained_correct,

        "base_correct_to_verifier_wrong":
            introduced_errors,

        "base_correct_to_abstain":
            over_abstentions,

        "base_wrong_to_verifier_correct":
            repaired_errors,

        "base_wrong_to_verifier_wrong":
            residual_errors,

        "base_wrong_to_abstain":
            good_catches,

        "abstention_recall_base_wrong":
            abstention_recall,

        "over_abstention_rate_base_correct":
            over_abstention_rate,

        "baseline_error_repair_rate":
            repair_rate,
    },


    "stop_reasons":
        dict(
            stop_reasons
        ),


    "interventions":
        interventions,


    "records":
        judged_records,
}


with open(
    FINAL_OUT,
    "w",
) as f:

    json.dump(
        final_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print("=" * 80)

print(
    "FINAL TEST SAVED:"
)

print(
    FINAL_OUT
)



print()
print("=" * 80)
print("WRONG COMMITTED TEST ANSWERS")
print("=" * 80)


for r in wrong_commits:

    print()

    print(
        f"qi={r['question_index']}"
    )

    print(
        "Q:",
        r["question"]
    )

    print(
        "Prediction:",
        r["prediction"]
    )

    print(
        "Gold:",
        r["gold_answer"]
    )

FINAL FROZEN TEST
Test questions: 200
Unique IDs: 200
Validation/test overlap: 0


RUNNING FROZEN VERIFIER ON TEST SET

[001/200] qi=776
Q: Are Roger Waters and Tom Johnston both musicians?
  outcome: commit
  reason: answer_claim_verified
  prediction: Yes

[002/200] qi=507
Q: What is the name for the adventure in "Tunnels and Trolls", a game designed by Ken St. Andre?
  outcome: stop
  reason: max_steps_without_verified_answer_claim
  ABSTAIN

[003/200] qi=895
Q: What football club plays in the area between the old tool gates: Brook Bar and Trafford bar?
  outcome: commit
  reason: answer_claim_verified
  prediction: Manchester United F.C

[004/200] qi=922
Q: The Russian route M9 forms a part of what European route that has a length of about 5320 km?
  outcome: commit
  reason: answer_claim_verified
  prediction: E22

[005/200] qi=33
Q: College Humor is a 1933 American pre-Code musical comedy film that starred what American singer and actor who has a trademark warm
  outcome: stop
  

In [ ]:


import json

DATA_DIR = "/content/drive/MyDrive/hedge_run"

FINAL_JSON = (
    f"{DATA_DIR}/verifier_full_claim_FINAL_TEST_judged.json"
)

OUT_JSON = (
    f"{DATA_DIR}/verifier_full_claim_FINAL_TEST_comparison_metrics.json"
)

with open(FINAL_JSON) as f:
    d = json.load(f)

m = d["metrics"]
b = d["baseline_same_test"]
t = d["transitions"]

N = m["n"]



committed = m["committed"]
correct = m["correct_committed"]
wrong = m["wrong_committed"]

base_correct = b["correct"]
base_wrong = b["wrong"]

over_abstained = t["base_correct_to_abstain"]
caught_wrong = t["base_wrong_to_abstain"]



coverage = committed / N

selective_accuracy = (
    correct / committed
    if committed else 0.0
)

confident_error_rate = wrong / N

over_abstention = (
    over_abstained / base_correct
    if base_correct else 0.0
)

abstention_recall = (
    caught_wrong / base_wrong
    if base_wrong else 0.0
)



pc0 = base_correct / N
pw0 = base_wrong / N

pc = correct / N
pw = wrong / N

THSx100 = (
    ((pc * pw0 - pw * pc0) / pw0) * 100
    if pw0 > 0
    else 0.0
)


overall_accuracy = correct / N

wrong_answer_rate_answered = (
    wrong / committed
    if committed else 0.0
)

repair_rate = (
    t["base_wrong_to_verifier_correct"] / base_wrong
    if base_wrong else 0.0
)

base_correct_retention = (
    t["base_correct_to_verifier_correct"] / base_correct
    if base_correct else 0.0
)

introduced_error_rate = (
    t["base_correct_to_verifier_wrong"] / base_correct
    if base_correct else 0.0
)



result = {
    "coverage": coverage,
    "selective_accuracy": selective_accuracy,
    "confident_error_rate": confident_error_rate,
    "over_abstention": over_abstention,
    "abstention_recall": abstention_recall,
    "THSx100": THSx100,

    "overall_semantic_accuracy": overall_accuracy,
    "wrong_answer_rate_answered": wrong_answer_rate_answered,
    "baseline_error_repair_rate": repair_rate,
    "base_correct_retention_rate": base_correct_retention,
    "introduced_error_rate": introduced_error_rate,
}

with open(OUT_JSON, "w") as f:
    json.dump(result, f, indent=2)

print("=" * 70)
print("FINAL HOTPOT COMPARISON METRICS")
print("=" * 70)

for key in [
    "coverage",
    "selective_accuracy",
    "confident_error_rate",
    "over_abstention",
    "abstention_recall",
    "THSx100",
]:
    print(f"{key:28s} {result[key]:.3f}")

print("\nExtra metrics:")

for key in [
    "overall_semantic_accuracy",
    "wrong_answer_rate_answered",
    "baseline_error_repair_rate",
    "base_correct_retention_rate",
    "introduced_error_rate",
]:
    print(f"{key:28s} {result[key]:.3f}")

print("\nSaved:")
print(OUT_JSON)

FINAL HOTPOT COMPARISON METRICS
coverage                     0.710
selective_accuracy           0.761
confident_error_rate         0.170
over_abstention              0.270
abstention_recall            0.302
THSx100                      44.016

Extra metrics:
overall_semantic_accuracy    0.540
wrong_answer_rate_answered   0.239
baseline_error_repair_rate   0.492
base_correct_retention_rate  0.622
introduced_error_rate        0.108

Saved:
/content/drive/MyDrive/hedge_run/verifier_full_claim_FINAL_TEST_comparison_metrics.json


In [ ]:


import os
import re
import json
import copy
import random
import numpy as np
import torch

from collections import Counter
from openai import OpenAI



required = [
    "run_question_full_claim",
    "lm",
    "tok",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Frozen full-claim verifier is not loaded.\n"
        f"Missing: {missing}\n\n"
        "Run the final verifier definition cell first."
    )



DATA_DIR = "/content/drive/MyDrive/hedge_run"

FULL_JSON = (
    f"{DATA_DIR}/2wiki_full.json"
)

TRAIN_JSON = (
    f"{DATA_DIR}/2wiki_train_questions.json"
)

TEST_JSON = (
    f"{DATA_DIR}/2wiki_test_questions.json"
)

BASE_JUDGED = (
    f"{DATA_DIR}/2wiki_base_judged.json"
)


TUNE_DIR = (
    f"{DATA_DIR}/2wiki_verifier_tuning"
)

os.makedirs(
    TUNE_DIR,
    exist_ok=True,
)


JUDGE_CACHE = (
    f"{TUNE_DIR}/semantic_judge_cache.json"
)

SUMMARY_FILE = (
    f"{TUNE_DIR}/tuning_summary.json"
)

BEST_CONFIG_FILE = (
    f"{TUNE_DIR}/BEST_CONFIG.json"
)



N_VAL = 150

SEED = 42


# THS baseline value used for the 2Wiki comparisons.
THS_BASE_ACC = 0.440


CONFIGS = [

    {
        "name":
            "current_s6_c050",

        "MAX_STEPS":
            6,

        "CLAIM_SUPPORT_THR":
            0.50,
    },


    {
        "name":
            "s8_c040",

        "MAX_STEPS":
            8,

        "CLAIM_SUPPORT_THR":
            0.40,
    },


    {
        "name":
            "s8_c045",

        "MAX_STEPS":
            8,

        "CLAIM_SUPPORT_THR":
            0.45,
    },


    {
        "name":
            "s8_c050",

        "MAX_STEPS":
            8,

        "CLAIM_SUPPORT_THR":
            0.50,
    },


    {
        "name":
            "s8_c055",

        "MAX_STEPS":
            8,

        "CLAIM_SUPPORT_THR":
            0.55,
    },


    {
        "name":
            "s8_c060",

        "MAX_STEPS":
            8,

        "CLAIM_SUPPORT_THR":
            0.60,
    },
]




try:

    from google.colab import userdata

    OPENAI_API_KEY = userdata.get(
        "OPENAI_API_KEY"
    )

except Exception:

    OPENAI_API_KEY = os.getenv(
        "OPENAI_API_KEY"
    )


if not OPENAI_API_KEY:

    raise RuntimeError(
        "OPENAI_API_KEY not found."
    )


oai = OpenAI(
    api_key=OPENAI_API_KEY
)




for path in [
    FULL_JSON,
    TRAIN_JSON,
    TEST_JSON,
    BASE_JUDGED,
]:

    if not os.path.exists(
        path
    ):

        raise FileNotFoundError(
            path
        )


with open(
    FULL_JSON
) as f:

    full_raw = json.load(f)


full = {

    int(r["question_index"]):
        r

    for r in full_raw
}


with open(
    TRAIN_JSON
) as f:

    train_manifest = json.load(f)


train_ids = [

    int(x["question_index"])

    for x in train_manifest
]


with open(
    TEST_JSON
) as f:

    test_manifest = json.load(f)


test_ids = {

    int(x["question_index"])

    for x in test_manifest
}



val_ids = [

    qi

    for qi in train_ids

    if (
        qi in full
        and
        qi not in test_ids
    )

][
    :N_VAL
]


assert len(
    val_ids
) == N_VAL


assert not (
    set(val_ids)
    &
    test_ids
)


print("=" * 80)
print("2WIKI VALIDATION TUNING")
print("=" * 80)

print(
    "Validation questions:",
    len(val_ids)
)

print(
    "Test overlap:",
    len(
        set(val_ids)
        &
        test_ids
    )
)




with open(
    BASE_JUDGED
) as f:

    baseline_judged_raw = json.load(f)


baseline_judged = {

    int(x["question_index"]):
        x

    for x in baseline_judged_raw
}


missing_baseline = [

    qi

    for qi in val_ids

    if qi not in baseline_judged
]


if missing_baseline:

    raise RuntimeError(
        f"{len(missing_baseline)} validation questions "
        "are missing from 2wiki_base_judged.json.\n"
        f"First missing IDs: {missing_baseline[:10]}"
    )


val_base_correct = {

    qi:
        bool(
            baseline_judged[
                qi
            ].get(
                "judge_correct",
                False,
            )
        )

    for qi in val_ids
}


n_val_base_correct = sum(
    val_base_correct.values()
)


n_val_base_wrong = (
    N_VAL
    -
    n_val_base_correct
)


val_base_accuracy = (
    n_val_base_correct
    /
    N_VAL
)


print()
print(
    "Validation baseline correct:",
    n_val_base_correct
)

print(
    "Validation baseline wrong:",
    n_val_base_wrong
)

print(
    f"Validation baseline accuracy: "
    f"{val_base_accuracy:.3f}"
)

print(
    "THS reference baseline:",
    THS_BASE_ACC
)




def get_gold(
    rec
):

    for key in [
        "gold_answer",
        "answer",
        "gold",
    ]:

        value = rec.get(
            key
        )


        if (
            value is not None
            and
            str(value).strip()
        ):

            return str(
                value
            ).strip()


    raise KeyError(
        "No gold answer for "
        f"qi={rec.get('question_index')}"
    )




def adapt_2wiki_record(
    record
):

    rec = copy.deepcopy(
        record
    )


    context = rec.get(
        "context",
        {}
    )


    if isinstance(
        context,
        dict,
    ):

        groups = context.get(
            "sentences",
            [],
        )


        if (
            groups
            and
            isinstance(
                groups[0],
                str,
            )
        ):

            groups = [
                groups
            ]


        titles = (
            context.get(
                "title"
            )
            or
            context.get(
                "titles"
            )
        )


        if titles is None:

            titles = [

                f"Passage {i + 1}"

                for i in range(
                    len(groups)
                )
            ]


        elif isinstance(
            titles,
            str,
        ):

            titles = [
                titles
            ]


        titles = list(
            titles
        )


        if len(titles) < len(groups):

            titles.extend(

                [

                    f"Passage {i + 1}"

                    for i in range(
                        len(titles),
                        len(groups),
                    )
                ]
            )


        if len(titles) > len(groups):

            titles = titles[
                :len(groups)
            ]


        rec[
            "context"
        ] = {

            **context,

            "title":
                titles,

            "sentences":
                groups,
        }


    else:

        raise ValueError(
            "Unexpected 2Wiki context format."
        )


    return rec



def seed_question(
    qi
):

    seed = (
        SEED
        +
        int(qi)
    )


    random.seed(
        seed
    )

    np.random.seed(
        seed
        %
        (2**32 - 1)
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )


def judge_answer(
    question,
    gold,
    prediction,
):

    if (
        not prediction
        or
        not str(
            prediction
        ).strip()
    ):

        return False, ""


    prompt = (

        f"Question: {question}\n"

        f"Correct answer: {gold}\n"

        f"Model's answer: {prediction}\n\n"

        "Correct? Accept paraphrases. "
        "End 'Verdict: yes' or 'Verdict: no'."
    )


    response = (
        oai.chat.completions.create(

            model=
                "gpt-4o-mini",

            temperature=
                0,

            max_tokens=
                120,

            messages=[
                {
                    "role":
                        "user",

                    "content":
                        prompt,
                }
            ],
        )
    )


    raw = (
        response
        .choices[0]
        .message.content
        .strip()
    )


    match = re.search(
        r"verdict:\s*(yes|no)",
        raw.lower(),
    )


    correct = bool(
        match
        and
        match.group(1)
        ==
        "yes"
    )


    return correct, raw




if os.path.exists(
    JUDGE_CACHE
):

    with open(
        JUDGE_CACHE
    ) as f:

        judge_cache = json.load(f)

else:

    judge_cache = {}


def cached_judge(
    qi,
    question,
    gold,
    prediction,
):



    cache_key = (
        f"{qi}::"
        f"{prediction}"
    )


    cached = judge_cache.get(
        cache_key
    )


    if (
        cached is not None
        and
        cached.get(
            "gold"
        )
        ==
        gold
    ):

        return (
            bool(
                cached[
                    "correct"
                ]
            ),

            cached.get(
                "raw",
                "",
            ),

            "cached",
        )


    correct, raw = judge_answer(
        question,
        gold,
        prediction,
    )


    judge_cache[
        cache_key
    ] = {

        "question_index":
            int(qi),

        "prediction":
            prediction,

        "gold":
            gold,

        "correct":
            bool(
                correct
            ),

        "raw":
            raw,
    }


    with open(
        JUDGE_CACHE,
        "w",
    ) as f:

        json.dump(
            judge_cache,
            f,
            indent=2,
            ensure_ascii=False,
        )


    return (
        correct,
        raw,
        "new",
    )




def calculate_ths(
    correct_committed,
    wrong_committed,
    N,
):

    pc = (
        correct_committed
        /
        N
    )


    pw = (
        wrong_committed
        /
        N
    )


    base = (
        THS_BASE_ACC
    )


    return (

        (
            pc
            *
            (1 - base)

            -

            pw
            *
            base
        )

        /

        (1 - base)

        *

        100
    )




def evaluate_config(
    config
):

    global MAX_STEPS
    global CLAIM_SUPPORT_THR


    MAX_STEPS = int(
        config[
            "MAX_STEPS"
        ]
    )


    CLAIM_SUPPORT_THR = float(
        config[
            "CLAIM_SUPPORT_THR"
        ]
    )


    name = config[
        "name"
    ]


    config_dir = os.path.join(
        TUNE_DIR,
        name,
    )


    os.makedirs(
        config_dir,
        exist_ok=True,
    )


    progress_file = os.path.join(
        config_dir,
        "progress.json",
    )


    final_file = os.path.join(
        config_dir,
        "result.json",
    )



    records = []


    if os.path.exists(
        progress_file
    ):

        with open(
            progress_file
        ) as f:

            progress = json.load(f)


        if (
            progress.get(
                "validation_ids"
            )
            !=
            val_ids
        ):

            raise RuntimeError(
                f"{name}: validation IDs changed."
            )


        saved_config = progress.get(
            "config",
            {}
        )


        if (
            saved_config.get(
                "MAX_STEPS"
            )
            !=
            MAX_STEPS

            or

            float(
                saved_config.get(
                    "CLAIM_SUPPORT_THR",
                    -1,
                )
            )
            !=
            CLAIM_SUPPORT_THR
        ):

            raise RuntimeError(
                f"{name}: checkpoint configuration mismatch."
            )


        records = progress.get(
            "records",
            []
        )


    done_ids = {

        int(
            r[
                "question_index"
            ]
        )

        for r in records
    }


    print()
    print("=" * 80)

    print(
        f"CONFIG: {name}"
    )

    print(
        f"MAX_STEPS={MAX_STEPS} | "
        f"CLAIM_SUPPORT_THR={CLAIM_SUPPORT_THR}"
    )

    print(
        f"resume={len(records)}/{N_VAL}"
    )

    print("=" * 80)




    for position, qi in enumerate(
        val_ids,
        start=1,
    ):

        if qi in done_ids:
            continue


        raw_rec = full[
            qi
        ]


        rec = adapt_2wiki_record(
            raw_rec
        )


        question = str(
            rec[
                "question"
            ]
        ).strip()


        gold = get_gold(
            raw_rec
        )


        seed_question(
            qi
        )


        print(
            f"[{position:03d}/{N_VAL}] "
            f"qi={qi}",
            end=" ",
        )


        try:

            output = (
                run_question_full_claim(
                    rec,
                    question,
                )
            )


        except Exception as exc:

            print(
                "ERROR:",
                type(exc).__name__,
                str(exc),
            )


            output = {

                "kind":
                    "error",

                "answer":
                    "",

                "reason":
                    (
                        f"{type(exc).__name__}: "
                        f"{str(exc)}"
                    ),
            }


        prediction = (

            str(
                output.get(
                    "answer",
                    "",
                )
            ).strip()

            if (
                output.get(
                    "kind"
                )
                ==
                "commit"
            )

            else ""
        )


        if (
            output.get(
                "kind"
            )
            ==
            "commit"
        ):

            correct, judge_raw, source = (
                cached_judge(
                    qi,
                    question,
                    gold,
                    prediction,
                )
            )


            print(
                f"{'✓' if correct else '✗'} "
                f"{prediction[:55]}"
            )


        elif (
            output.get(
                "kind"
            )
            ==
            "error"
        ):

            correct = None
            judge_raw = None

            print(
                "ERROR"
            )


        else:

            correct = None
            judge_raw = None

            print(
                "ABSTAIN"
            )


        records.append({

            "question_index":
                qi,

            "question":
                question,

            "gold_answer":
                gold,

            "baseline_correct":
                bool(
                    val_base_correct[
                        qi
                    ]
                ),

            "prediction":
                prediction,

            "verifier_correct":
                correct,

            "judge_raw":
                judge_raw,

            "output":
                output,
        })


        # checkpoint every question

        with open(
            progress_file,
            "w",
        ) as f:

            json.dump(
                {
                    "config":
                        config,

                    "validation_ids":
                        val_ids,

                    "records":
                        records,
                },
                f,
                indent=2,
                ensure_ascii=False,
            )



    valid = [

        r

        for r in records

        if r[
            "output"
        ].get(
            "kind"
        )
        !=
        "error"
    ]


    N = len(
        valid
    )


    if N != N_VAL:

        raise RuntimeError(
            f"{name}: only {N}/{N_VAL} valid questions."
        )


    committed = [

        r

        for r in valid

        if (
            r[
                "output"
            ].get(
                "kind"
            )
            ==
            "commit"
        )
    ]


    abstained = [

        r

        for r in valid

        if (
            r[
                "output"
            ].get(
                "kind"
            )
            !=
            "commit"
        )
    ]


    correct_committed = sum(

        bool(
            r[
                "verifier_correct"
            ]
        )

        for r in committed
    )


    wrong_committed = (

        len(
            committed
        )

        -
        correct_committed
    )



    transition = Counter()


    for r in valid:

        base_state = (

            "base_correct"

            if r[
                "baseline_correct"
            ]

            else
            "base_wrong"
        )


        if (
            r[
                "output"
            ].get(
                "kind"
            )
            !=
            "commit"
        ):

            verifier_state = (
                "abstain"
            )


        elif r[
            "verifier_correct"
        ]:

            verifier_state = (
                "verifier_correct"
            )


        else:

            verifier_state = (
                "verifier_wrong"
            )


        transition[
            (
                base_state,
                verifier_state,
            )
        ] += 1


    BC_VC = transition[
        (
            "base_correct",
            "verifier_correct",
        )
    ]


    BC_VW = transition[
        (
            "base_correct",
            "verifier_wrong",
        )
    ]


    BC_A = transition[
        (
            "base_correct",
            "abstain",
        )
    ]


    BW_VC = transition[
        (
            "base_wrong",
            "verifier_correct",
        )
    ]


    BW_VW = transition[
        (
            "base_wrong",
            "verifier_wrong",
        )
    ]


    BW_A = transition[
        (
            "base_wrong",
            "abstain",
        )
    ]



    coverage = (
        len(committed)
        /
        N
    )


    selective_accuracy = (

        correct_committed
        /
        len(committed)

        if committed

        else 0.0
    )


    confident_error_rate = (
        wrong_committed
        /
        N
    )


    over_abstention = (

        BC_A
        /
        n_val_base_correct

        if n_val_base_correct

        else 0.0
    )


    abstention_recall = (

        BW_A
        /
        n_val_base_wrong

        if n_val_base_wrong

        else 0.0
    )


    THSx100 = calculate_ths(
        correct_committed,
        wrong_committed,
        N,
    )


    overall_accuracy = (
        correct_committed
        /
        N
    )


    repair_rate = (

        BW_VC
        /
        n_val_base_wrong

        if n_val_base_wrong

        else 0.0
    )


    introduced_error_rate = (

        BC_VW
        /
        n_val_base_correct

        if n_val_base_correct

        else 0.0
    )


    stop_reasons = Counter(

        r[
            "output"
        ].get(
            "reason"
        )

        for r in abstained
    )


    metrics = {

        "name":
            name,

        "MAX_STEPS":
            MAX_STEPS,

        "CLAIM_SUPPORT_THR":
            CLAIM_SUPPORT_THR,

        "N":
            N,

        "committed":
            len(committed),

        "abstained":
            len(abstained),

        "correct_committed":
            correct_committed,

        "wrong_committed":
            wrong_committed,

        "coverage":
            coverage,

        "selective_accuracy":
            selective_accuracy,

        "confident_error_rate":
            confident_error_rate,

        "over_abstention":
            over_abstention,

        "abstention_recall":
            abstention_recall,

        "THSx100":
            THSx100,

        "overall_semantic_accuracy":
            overall_accuracy,

        "baseline_error_repair_rate":
            repair_rate,

        "introduced_error_rate":
            introduced_error_rate,

        "stop_reasons":
            dict(
                stop_reasons
            ),

        "transitions": {

            "base_correct_to_verifier_correct":
                BC_VC,

            "base_correct_to_verifier_wrong":
                BC_VW,

            "base_correct_to_abstain":
                BC_A,

            "base_wrong_to_verifier_correct":
                BW_VC,

            "base_wrong_to_verifier_wrong":
                BW_VW,

            "base_wrong_to_abstain":
                BW_A,
        },
    }


    with open(
        final_file,
        "w",
    ) as f:

        json.dump(
            {
                "config":
                    config,

                "metrics":
                    metrics,

                "records":
                    records,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )


    print()
    print(
        f"{name}:"
    )

    print(
        f"  coverage          = "
        f"{coverage:.3f}"
    )

    print(
        f"  selective acc.    = "
        f"{selective_accuracy:.3f}"
    )

    print(
        f"  confident error   = "
        f"{confident_error_rate:.3f}"
    )

    print(
        f"  over-abstention   = "
        f"{over_abstention:.3f}"
    )

    print(
        f"  abstention recall = "
        f"{abstention_recall:.3f}"
    )

    print(
        f"  THS               = "
        f"{THSx100:.2f}"
    )


    return metrics




all_results = []


for config in CONFIGS:

    result = evaluate_config(
        config
    )

    all_results.append(
        result
    )




all_results = sorted(

    all_results,

    key=lambda r: (

        r[
            "THSx100"
        ],

        -
        r[
            "confident_error_rate"
        ],

        r[
            "coverage"
        ],
    ),

    reverse=True,
)


best = all_results[
    0
]




print()
print("=" * 110)

print(
    "2WIKI VALIDATION TUNING RESULTS"
)

print("=" * 110)


header = (

    f"{'Config':18s} "
    f"{'Steps':>6s} "
    f"{'ClaimThr':>9s} "
    f"{'Cov':>7s} "
    f"{'SelAcc':>8s} "
    f"{'ConfErr':>8s} "
    f"{'OverAb':>8s} "
    f"{'AbsRec':>8s} "
    f"{'THS':>8s}"
)

print(
    header
)

print(
    "-" * len(
        header
    )
)


for r in all_results:

    print(

        f"{r['name']:18s} "

        f"{r['MAX_STEPS']:6d} "

        f"{r['CLAIM_SUPPORT_THR']:9.2f} "

        f"{r['coverage']:7.3f} "

        f"{r['selective_accuracy']:8.3f} "

        f"{r['confident_error_rate']:8.3f} "

        f"{r['over_abstention']:8.3f} "

        f"{r['abstention_recall']:8.3f} "

        f"{r['THSx100']:8.2f}"
    )




print()
print("=" * 80)
print("SELECTED 2WIKI CONFIGURATION")
print("=" * 80)


print(
    f"Name:               "
    f"{best['name']}"
)

print(
    f"MAX_STEPS:          "
    f"{best['MAX_STEPS']}"
)

print(
    f"CLAIM_SUPPORT_THR:  "
    f"{best['CLAIM_SUPPORT_THR']}"
)

print()

print(
    f"Validation coverage:       "
    f"{best['coverage']:.3f}"
)

print(
    f"Validation selective acc.: "
    f"{best['selective_accuracy']:.3f}"
)

print(
    f"Validation confident err.: "
    f"{best['confident_error_rate']:.3f}"
)

print(
    f"Validation over-abstention:"
    f" {best['over_abstention']:.3f}"
)

print(
    f"Validation abst. recall:   "
    f"{best['abstention_recall']:.3f}"
)

print(
    f"Validation THS:            "
    f"{best['THSx100']:.2f}"
)



summary = {

    "dataset":
        "2WikiMultihopQA",

    "selection_split":
        "first_150_from_2wiki_train_questions",

    "n_validation":
        N_VAL,

    "test_overlap":
        len(
            set(val_ids)
            &
            test_ids
        ),

    "validation_baseline": {

        "correct":
            n_val_base_correct,

        "wrong":
            n_val_base_wrong,

        "accuracy":
            val_base_accuracy,

        "THS_reference_base_accuracy":
            THS_BASE_ACC,
    },

    "all_results":
        all_results,

    "best":
        best,
}


with open(
    SUMMARY_FILE,
    "w",
) as f:

    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False,
    )


best_config = {

    "MAX_STEPS":
        best[
            "MAX_STEPS"
        ],

    "CLAIM_SUPPORT_THR":
        best[
            "CLAIM_SUPPORT_THR"
        ],

    # All remaining verifier parameters stay unchanged.
}


with open(
    BEST_CONFIG_FILE,
    "w",
) as f:

    json.dump(
        best_config,
        f,
        indent=2,
    )


print()
print(
    "Saved tuning summary:"
)

print(
    SUMMARY_FILE
)


print()
print(
    "Saved config to freeze:"
)

print(
    BEST_CONFIG_FILE
)




print()
print("=" * 80)
print("USE THESE SETTINGS FOR THE FINAL 2WIKI TEST")
print("=" * 80)

print(
    f"MAX_STEPS = "
    f"{best['MAX_STEPS']}"
)

print(
    f"CLAIM_SUPPORT_THR = "
    f"{best['CLAIM_SUPPORT_THR']}"
)

print()
print(
    "Do not change any other verifier parameter."
)

2WIKI VALIDATION TUNING
Validation questions: 150
Test overlap: 0

Validation baseline correct: 66
Validation baseline wrong: 84
Validation baseline accuracy: 0.440
THS reference baseline: 0.44

CONFIG: current_s6_c050
MAX_STEPS=6 | CLAIM_SUPPORT_THR=0.5
resume=0/150
[001/150] qi=8867 ABSTAIN
[002/150] qi=5251 ✓ Hamlet Gonashvili
[003/150] qi=8917 ABSTAIN
[004/150] qi=11966 ✓ 1510
[005/150] qi=3449 ✓ Hafez al-Assad
[006/150] qi=8195 ✓ Mongol Empire
[007/150] qi=7464 ✓ Łódź
[008/150] qi=10943 ✓ Orlando, Florida
[009/150] qi=5418 ABSTAIN
[010/150] qi=6366 ABSTAIN
[011/150] qi=3095 ABSTAIN
[012/150] qi=1974 ABSTAIN
[013/150] qi=3671 ABSTAIN
[014/150] qi=6094 ✓ Boston, Massachusetts
[015/150] qi=4848 ✓ New York City
[016/150] qi=3616 ABSTAIN
[017/150] qi=5410 ✓ England
[018/150] qi=687 ABSTAIN
[019/150] qi=7554 ✓ American
[020/150] qi=2598 ✗ Elias Kazantzoglou
[021/150] qi=3154 ✗ Maria of Calabria
[022/150] qi=8389 ABSTAIN
[023/150] qi=371 ✓ Dialogues Of Exiles
[024/150] qi=277 ✓ US
[025/1

In [ ]:


import os
import re
import json
import copy
from collections import Counter

from openai import OpenAI



required = [
    "run_question_full_claim",
    "lm",
    "tok",
]

missing = [
    x for x in required
    if x not in globals()
]

if missing:
    raise RuntimeError(
        "Frozen verifier is not loaded.\n"
        f"Missing: {missing}"
    )



MAX_STEPS = 8
CLAIM_SUPPORT_THR = 0.55

print("=" * 80)
print("2WIKI TUNED FINAL TEST")
print("=" * 80)

print("MAX_STEPS =", MAX_STEPS)
print("CLAIM_SUPPORT_THR =", CLAIM_SUPPORT_THR)


DATA_DIR = "/content/drive/MyDrive/hedge_run"

FULL_JSON = (
    f"{DATA_DIR}/2wiki_full.json"
)

TEST_JSON = (
    f"{DATA_DIR}/2wiki_test_questions.json"
)

FAIR_JSON = (
    f"{DATA_DIR}/2wiki_base_point_fair.json"
)





PROGRESS_FILE = (
    f"{DATA_DIR}/2wiki_TUNED_s8_c055_progress.json"
)

RAW_FILE = (
    f"{DATA_DIR}/2wiki_TUNED_s8_c055_raw.json"
)

JUDGE_CACHE = (
    f"{DATA_DIR}/2wiki_TUNED_s8_c055_judge_cache.json"
)

FINAL_FILE = (
    f"{DATA_DIR}/2wiki_TUNED_s8_c055_FINAL.json"
)



try:
    from google.colab import userdata

    OPENAI_API_KEY = userdata.get(
        "OPENAI_API_KEY"
    )

except Exception:

    OPENAI_API_KEY = os.getenv(
        "OPENAI_API_KEY"
    )


if not OPENAI_API_KEY:
    raise RuntimeError(
        "OPENAI_API_KEY not found."
    )


oai = OpenAI(
    api_key=OPENAI_API_KEY
)



for path in [
    FULL_JSON,
    TEST_JSON,
    FAIR_JSON,
]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)


with open(FULL_JSON) as f:
    full_raw = json.load(f)


full = {
    int(r["question_index"]): r
    for r in full_raw
}


with open(TEST_JSON) as f:
    test_manifest = json.load(f)


test_ids = [
    int(x["question_index"])
    for x in test_manifest
]


with open(FAIR_JSON) as f:
    fair = json.load(f)


base_correct_q = {
    int(k): bool(v)
    for k, v in fair["per_q"].items()
}



assert len(test_ids) == 200
assert len(set(test_ids)) == 200


missing_full = [
    qi
    for qi in test_ids
    if qi not in full
]

assert not missing_full, (
    f"Missing test questions: {missing_full[:10]}"
)


missing_baseline = [
    qi
    for qi in test_ids
    if qi not in base_correct_q
]

assert not missing_baseline, (
    f"Missing baseline labels: {missing_baseline[:10]}"
)


n_base_correct_expected = sum(
    base_correct_q[qi]
    for qi in test_ids
)

n_base_wrong_expected = (
    len(test_ids)
    -
    n_base_correct_expected
)


print()
print("Test questions:", len(test_ids))

print(
    "Baseline correct:",
    n_base_correct_expected
)

print(
    "Baseline wrong:",
    n_base_wrong_expected
)

print(
    "Baseline accuracy:",
    n_base_correct_expected / len(test_ids)
)


assert n_base_correct_expected == 88
assert n_base_wrong_expected == 112


print()
print("✓ Exact same 200-question 2Wiki test split")
print("✓ Baseline = 88/200 = 0.440")
print("✓ Tuned parameters frozen")
print()


def adapt_2wiki_record(record):

    rec = copy.deepcopy(record)

    context = rec.get(
        "context",
        {}
    )


    if not isinstance(
        context,
        dict,
    ):
        raise ValueError(
            f"Unexpected context format for "
            f"qi={rec.get('question_index')}"
        )


    groups = context.get(
        "sentences",
        [],
    )


    if (
        groups
        and
        isinstance(
            groups[0],
            str,
        )
    ):
        groups = [groups]


    titles = (
        context.get("title")
        or
        context.get("titles")
    )


    if titles is None:

        titles = [
            f"Passage {i + 1}"
            for i in range(
                len(groups)
            )
        ]


    elif isinstance(
        titles,
        str,
    ):

        titles = [titles]


    titles = list(titles)


    if len(titles) < len(groups):

        titles.extend(
            [
                f"Passage {i + 1}"
                for i in range(
                    len(titles),
                    len(groups),
                )
            ]
        )


    elif len(titles) > len(groups):

        titles = titles[
            :len(groups)
        ]


    rec["context"] = {
        **context,

        "title":
            titles,

        "sentences":
            groups,
    }


    return rec




def get_gold(rec):

    for key in [
        "gold_answer",
        "answer",
        "gold",
    ]:

        value = rec.get(key)

        if (
            value is not None
            and
            str(value).strip()
        ):
            return str(value).strip()


    raise KeyError(
        f"No gold answer for "
        f"qi={rec.get('question_index')}"
    )




records = []


if os.path.exists(
    PROGRESS_FILE
):

    with open(
        PROGRESS_FILE
    ) as f:

        progress = json.load(f)


    saved_ids = [
        int(x)
        for x in progress.get(
            "test_ids",
            []
        )
    ]


    if saved_ids != test_ids:

        raise RuntimeError(
            "Existing tuned checkpoint uses "
            "a different test split."
        )


    saved_config = progress.get(
        "config",
        {}
    )


    if (
        saved_config.get(
            "MAX_STEPS"
        )
        !=
        MAX_STEPS
        or
        float(
            saved_config.get(
                "CLAIM_SUPPORT_THR",
                -1
            )
        )
        !=
        CLAIM_SUPPORT_THR
    ):

        raise RuntimeError(
            "Existing checkpoint has different "
            "tuned parameters."
        )


    records = progress.get(
        "records",
        []
    )


    print(
        f"Resuming tuned test: "
        f"{len(records)}/200 completed"
    )


done_ids = {
    int(r["question_index"])
    for r in records
}



print()
print("=" * 80)
print("RUNNING TUNED VERIFIER ON 2WIKI TEST")
print("=" * 80)


for position, qi in enumerate(
    test_ids,
    start=1,
):

    if qi in done_ids:
        continue


    original = full[qi]

    rec = adapt_2wiki_record(
        original
    )


    question = str(
        rec["question"]
    ).strip()


    gold = get_gold(
        original
    )


    print()
    print(
        f"[{position:03d}/200] "
        f"qi={qi}"
    )

    print(
        "Q:",
        question[:140]
    )




    try:

        output = run_question_full_claim(
            rec,
            question,
        )


    except Exception as exc:

        output = {
            "kind":
                "error",

            "answer":
                "",

            "reason":
                (
                    f"{type(exc).__name__}: "
                    f"{str(exc)}"
                ),
        }


        print(
            "  ERROR:",
            output["reason"]
        )


    if output.get(
        "kind"
    ) == "commit":

        print(
            "  COMMIT:",
            output.get(
                "answer",
                ""
            )
        )


    elif output.get(
        "kind"
    ) != "error":

        print("  ABSTAIN")

        print(
            "  reason:",
            output.get(
                "reason"
            )
        )




    records.append({

        "question_index":
            qi,

        "question":
            question,

        "gold_answer":
            gold,

        "baseline_correct":
            bool(
                base_correct_q[qi]
            ),

        "output":
            output,
    })




    with open(
        PROGRESS_FILE,
        "w",
    ) as f:

        json.dump(
            {
                "dataset":
                    "2WikiMultihopQA",

                "architecture":
                    "full_question_claim",

                "config": {
                    "MAX_STEPS":
                        MAX_STEPS,

                    "CLAIM_SUPPORT_THR":
                        CLAIM_SUPPORT_THR,
                },

                "test_ids":
                    test_ids,

                "records":
                    records,
            },
            f,
            indent=2,
            ensure_ascii=False,
        )



with open(
    RAW_FILE,
    "w",
) as f:

    json.dump(
        {
            "dataset":
                "2WikiMultihopQA",

            "architecture":
                "full_question_claim",

            "config": {
                "MAX_STEPS":
                    MAX_STEPS,

                "CLAIM_SUPPORT_THR":
                    CLAIM_SUPPORT_THR,
            },

            "n":
                len(records),

            "test_ids":
                test_ids,

            "records":
                records,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print(
    "Raw tuned test saved:"
)

print(
    RAW_FILE
)




def judge_answer(
    question,
    gold,
    prediction,
):

    if (
        not prediction
        or
        not str(
            prediction
        ).strip()
    ):
        return False, ""


    prompt = (
        f"Question: {question}\n"
        f"Correct answer: {gold}\n"
        f"Model's answer: {prediction}\n\n"
        "Correct? Accept paraphrases. "
        "End 'Verdict: yes' or 'Verdict: no'."
    )


    response = (
        oai.chat.completions.create(
            model=
                "gpt-4o-mini",

            temperature=
                0,

            max_tokens=
                120,

            messages=[
                {
                    "role":
                        "user",

                    "content":
                        prompt,
                }
            ],
        )
    )


    raw = (
        response
        .choices[0]
        .message.content
        .strip()
    )


    match = re.search(
        r"verdict:\s*(yes|no)",
        raw.lower(),
    )


    correct = bool(
        match
        and
        match.group(1) == "yes"
    )


    return correct, raw



if os.path.exists(
    JUDGE_CACHE
):

    with open(
        JUDGE_CACHE
    ) as f:

        judge_cache = json.load(f)


    print(
        "Loaded tuned judge cache:",
        len(judge_cache)
    )


else:

    judge_cache = {}



judged_records = []


print()
print("=" * 80)
print("SEMANTIC JUDGING — TUNED 2WIKI TEST")
print("=" * 80)


for i, rec in enumerate(
    records,
    start=1,
):

    qi = int(
        rec["question_index"]
    )

    question = rec[
        "question"
    ]

    gold = rec[
        "gold_answer"
    ]

    output = rec[
        "output"
    ]

    base_correct = bool(
        rec[
            "baseline_correct"
        ]
    )



    if output.get(
        "kind"
    ) != "commit":

        kind = (
            "error"
            if output.get(
                "kind"
            ) == "error"
            else "abstain"
        )


        judged_records.append({

            "question_index":
                qi,

            "question":
                question,

            "gold_answer":
                gold,

            "prediction":
                "",

            "baseline_correct":
                base_correct,

            "kind":
                kind,

            "judge_correct":
                None,

            "judge_raw":
                None,

            "stop_reason":
                output.get(
                    "reason"
                ),
        })


        print(
            f"[{i:03d}/200] "
            f"qi={qi} "
            f"{kind.upper()}"
        )


        continue




    prediction = str(
        output.get(
            "answer",
            ""
        )
    ).strip()


    cache_key = str(qi)


    cached = judge_cache.get(
        cache_key
    )


    if (
        cached is not None
        and
        cached.get(
            "prediction"
        ) == prediction
        and
        cached.get(
            "gold_answer"
        ) == gold
    ):

        correct = bool(
            cached[
                "judge_correct"
            ]
        )

        raw = cached.get(
            "judge_raw",
            ""
        )

        source = "cached"


    else:

        correct, raw = judge_answer(
            question,
            gold,
            prediction,
        )


        judge_cache[
            cache_key
        ] = {

            "question_index":
                qi,

            "prediction":
                prediction,

            "gold_answer":
                gold,

            "judge_correct":
                bool(correct),

            "judge_raw":
                raw,
        }


        with open(
            JUDGE_CACHE,
            "w",
        ) as f:

            json.dump(
                judge_cache,
                f,
                indent=2,
                ensure_ascii=False,
            )


        source = "new"


    judged_records.append({

        "question_index":
            qi,

        "question":
            question,

        "gold_answer":
            gold,

        "prediction":
            prediction,

        "baseline_correct":
            base_correct,

        "kind":
            (
                "commit_correct"
                if correct
                else "commit_wrong"
            ),

        "judge_correct":
            bool(correct),

        "judge_raw":
            raw,

        "stop_reason":
            None,
    })


    print(
        f"[{i:03d}/200] "
        f"qi={qi} "
        f"{'✓' if correct else '✗'} "
        f"{prediction[:70]} "
        f"({source})"
    )



errors = [
    r
    for r in judged_records
    if r["kind"] == "error"
]


if errors:

    raise RuntimeError(
        f"{len(errors)} test questions had runtime errors. "
        "Fix/resume them before reporting final metrics."
    )


valid_records = judged_records

N = len(
    valid_records
)


assert N == 200




committed = [
    r
    for r in valid_records
    if r["kind"] in {
        "commit_correct",
        "commit_wrong",
    }
]


abstained = [
    r
    for r in valid_records
    if r["kind"] == "abstain"
]


correct_commits = [
    r
    for r in committed
    if r["judge_correct"]
]


wrong_commits = [
    r
    for r in committed
    if not r["judge_correct"]
]


n_commit = len(
    committed
)

n_abstain = len(
    abstained
)

n_correct = len(
    correct_commits
)

n_wrong = len(
    wrong_commits
)




n_base_correct = sum(
    r["baseline_correct"]
    for r in valid_records
)

n_base_wrong = (
    N
    -
    n_base_correct
)


baseline_accuracy = (
    n_base_correct
    /
    N
)


assert n_base_correct == 88
assert n_base_wrong == 112



transition = Counter()


for r in valid_records:

    base_state = (
        "base_correct"
        if r["baseline_correct"]
        else "base_wrong"
    )


    if r["kind"] == "abstain":

        verifier_state = (
            "abstain"
        )


    elif r["judge_correct"]:

        verifier_state = (
            "verifier_correct"
        )


    else:

        verifier_state = (
            "verifier_wrong"
        )


    transition[
        (
            base_state,
            verifier_state,
        )
    ] += 1


BC_VC = transition[
    (
        "base_correct",
        "verifier_correct",
    )
]

BC_VW = transition[
    (
        "base_correct",
        "verifier_wrong",
    )
]

BC_A = transition[
    (
        "base_correct",
        "abstain",
    )
]

BW_VC = transition[
    (
        "base_wrong",
        "verifier_correct",
    )
]

BW_VW = transition[
    (
        "base_wrong",
        "verifier_wrong",
    )
]

BW_A = transition[
    (
        "base_wrong",
        "abstain",
    )
]



coverage = (
    n_commit / N
)


selective_accuracy = (
    n_correct / n_commit
    if n_commit
    else 0.0
)


confident_error_rate = (
    n_wrong / N
)


over_abstention = (
    BC_A / n_base_correct
    if n_base_correct
    else 0.0
)


abstention_recall = (
    BW_A / n_base_wrong
    if n_base_wrong
    else 0.0
)




pc0 = (
    n_base_correct / N
)

pw0 = (
    n_base_wrong / N
)

pc = (
    n_correct / N
)

pw = (
    n_wrong / N
)


THSx100 = (
    (
        (
            pc * pw0
            -
            pw * pc0
        )
        /
        pw0
    )
    * 100

    if pw0 > 0
    else 0.0
)




overall_accuracy = (
    n_correct / N
)


wrong_answer_rate_answered = (
    n_wrong / n_commit
    if n_commit
    else 0.0
)


repair_rate = (
    BW_VC / n_base_wrong
    if n_base_wrong
    else 0.0
)


base_correct_retention = (
    BC_VC / n_base_correct
    if n_base_correct
    else 0.0
)


introduced_error_rate = (
    BC_VW / n_base_correct
    if n_base_correct
    else 0.0
)




stop_reasons = Counter(
    r.get(
        "stop_reason"
    )
    for r in abstained
)



COUNTER_FIELDS = [
    "num_contradicted_rejected",
    "num_conflicted_rejected",
    "num_unclear_seen",
    "num_grounded_unclear_selected",
    "num_repetitions_rejected",
    "num_meta_rejected",
    "num_premature_final_rejected",
    "num_resamples",
    "num_successful_repairs",
    "answer_extraction_attempts",
    "answer_verification_failures",
    "claim_verification_attempts",
]


raw_by_qi = {
    int(r["question_index"]):
        r["output"]
    for r in records
}


interventions = {}


for field in COUNTER_FIELDS:

    interventions[
        field
    ] = int(
        sum(
            raw_by_qi[
                int(r["question_index"])
            ].get(
                field,
                0,
            )
            for r in valid_records
        )
    )




print()
print("=" * 80)
print("2WIKI TUNED FINAL TEST RESULTS")
print("=" * 80)

print(
    f"Test questions:                      {N}"
)

print(
    f"Committed:                           {n_commit}"
)

print(
    f"Abstained:                           {n_abstain}"
)

print(
    f"Coverage:                            {coverage:.3f}"
)


print()
print("--- Semantic correctness ---")

print(
    f"Correct committed:                   {n_correct}"
)

print(
    f"Wrong committed:                     {n_wrong}"
)

print(
    f"Selective semantic accuracy:         {selective_accuracy:.3f}"
)

print(
    f"Overall semantic accuracy:           {overall_accuracy:.3f}"
)

print(
    f"Wrong-answer rate / total:           {confident_error_rate:.3f}"
)

print(
    f"Wrong-answer rate / answered:        {wrong_answer_rate_answered:.3f}"
)




print()
print("--- BASELINE ON SAME 2WIKI TEST ---")

print(
    f"Baseline correct:                    {n_base_correct}"
)

print(
    f"Baseline wrong:                      {n_base_wrong}"
)

print(
    f"Baseline accuracy:                   {baseline_accuracy:.3f}"
)

print(
    f"Verifier selective accuracy:         {selective_accuracy:.3f}"
)

print(
    f"Selective accuracy gain:             "
    f"{selective_accuracy - baseline_accuracy:+.3f}"
)



print()
print("--- BASELINE -> VERIFIER TRANSITIONS ---")

print(
    f"Base correct -> verifier correct:    {BC_VC}"
)

print(
    f"Base correct -> verifier wrong:      {BC_VW}"
)

print(
    f"Base correct -> abstain:             {BC_A}"
)

print()

print(
    f"Base wrong   -> verifier correct:    {BW_VC}"
)

print(
    f"Base wrong   -> verifier wrong:      {BW_VW}"
)

print(
    f"Base wrong   -> abstain:             {BW_A}"
)




print()
print("=" * 80)
print("MAIN COMPARISON METRICS")
print("=" * 80)

print(
    f"coverage:                           {coverage:.3f}"
)

print(
    f"selective_accuracy:                 {selective_accuracy:.3f}"
)

print(
    f"confident_error_rate:               {confident_error_rate:.3f}"
)

print(
    f"over_abstention:                    {over_abstention:.3f}"
)

print(
    f"abstention_recall:                  {abstention_recall:.3f}"
)

print(
    f"THSx100:                            {THSx100:.3f}"
)



print()
print("--- EXTRA METRICS ---")

print(
    f"overall_semantic_accuracy:          {overall_accuracy:.3f}"
)

print(
    f"wrong_answer_rate_answered:         {wrong_answer_rate_answered:.3f}"
)

print(
    f"baseline_error_repair_rate:         {repair_rate:.3f}"
)

print(
    f"base_correct_retention_rate:        {base_correct_retention:.3f}"
)

print(
    f"introduced_error_rate:              {introduced_error_rate:.3f}"
)


print()
print("--- STOP REASONS ---")

for reason, count in (
    stop_reasons.items()
):
    print(
        f"{reason}: {count}"
    )


print()
print("--- VERIFIER INTERVENTIONS ---")

for key, value in (
    interventions.items()
):
    print(
        f"{key}: {value}"
    )


UNTUNED = {
    "coverage":
        0.575,

    "selective_accuracy":
        0.774,

    "confident_error_rate":
        0.130,

    "over_abstention":
        0.432,

    "abstention_recall":
        0.420,

    "THSx100":
        34.286,
}


print()
print("=" * 80)
print("TUNED vs UNTUNED 2WIKI TEST")
print("=" * 80)


comparison = {

    "coverage":
        coverage,

    "selective_accuracy":
        selective_accuracy,

    "confident_error_rate":
        confident_error_rate,

    "over_abstention":
        over_abstention,

    "abstention_recall":
        abstention_recall,

    "THSx100":
        THSx100,
}


for metric in [
    "coverage",
    "selective_accuracy",
    "confident_error_rate",
    "over_abstention",
    "abstention_recall",
    "THSx100",
]:

    old = UNTUNED[
        metric
    ]

    new = comparison[
        metric
    ]

    delta = (
        new - old
    )


    print(
        f"{metric:28s} "
        f"untuned={old:8.3f}  "
        f"tuned={new:8.3f}  "
        f"delta={delta:+8.3f}"
    )



final_payload = {

    "dataset":
        "2WikiMultihopQA",

    "version":
        "tuned",

    "architecture":
        "full_question_claim",

    "configuration": {

        "MAX_STEPS":
            MAX_STEPS,

        "CLAIM_SUPPORT_THR":
            CLAIM_SUPPORT_THR,

        "selected_on":
            "150-question 2Wiki validation set",

        "selection_metric":
            "THS",
    },


    "baseline": {

        "correct":
            n_base_correct,

        "wrong":
            n_base_wrong,

        "accuracy":
            baseline_accuracy,
    },


    "metrics": {

        "n":
            N,

        "committed":
            n_commit,

        "abstained":
            n_abstain,

        "coverage":
            coverage,

        "correct_committed":
            n_correct,

        "wrong_committed":
            n_wrong,

        "selective_accuracy":
            selective_accuracy,

        "overall_semantic_accuracy":
            overall_accuracy,

        "confident_error_rate":
            confident_error_rate,

        "wrong_answer_rate_answered":
            wrong_answer_rate_answered,

        "over_abstention":
            over_abstention,

        "abstention_recall":
            abstention_recall,

        "THSx100":
            THSx100,

        "baseline_error_repair_rate":
            repair_rate,

        "base_correct_retention_rate":
            base_correct_retention,

        "introduced_error_rate":
            introduced_error_rate,
    },


    "transitions": {

        "base_correct_to_verifier_correct":
            BC_VC,

        "base_correct_to_verifier_wrong":
            BC_VW,

        "base_correct_to_abstain":
            BC_A,

        "base_wrong_to_verifier_correct":
            BW_VC,

        "base_wrong_to_verifier_wrong":
            BW_VW,

        "base_wrong_to_abstain":
            BW_A,
    },


    "stop_reasons":
        dict(stop_reasons),


    "interventions":
        interventions,


    "untuned_test_reference":
        UNTUNED,


    "records":
        judged_records,
}


with open(
    FINAL_FILE,
    "w",
) as f:

    json.dump(
        final_payload,
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print("=" * 80)

print(
    "FINAL TUNED 2WIKI RESULT SAVED:"
)

print(
    FINAL_FILE
)




print()
print("=" * 80)
print("WRONG COMMITTED ANSWERS — TUNED TEST")
print("=" * 80)


for r in wrong_commits:

    print()

    print(
        f"qi={r['question_index']}"
    )

    print(
        "Q:",
        r["question"]
    )

    print(
        "Prediction:",
        r["prediction"]
    )

    print(
        "Gold:",
        r["gold_answer"]
    )

2WIKI TUNED FINAL TEST
MAX_STEPS = 8
CLAIM_SUPPORT_THR = 0.55

Test questions: 200
Baseline correct: 88
Baseline wrong: 112
Baseline accuracy: 0.44

✓ Exact same 200-question 2Wiki test split
✓ Baseline = 88/200 = 0.440
✓ Tuned parameters frozen


RUNNING TUNED VERIFIER ON 2WIKI TEST

[001/200] qi=2179
Q: Which film has the director who died earlier, Max And Helen or Held Einer Nacht?
  COMMIT: Held Einer Nacht

[002/200] qi=6748
Q: Who is younger, Jule Mallonee or Mimí Lazo?
  ABSTAIN
  reason: no_safe_grounded_continuation_after_retry

[003/200] qi=6133
Q: Who is Sophie Of France (1786-1787)'s maternal grandfather?
  COMMIT: Louis XVI of France

[004/200] qi=4229
Q: Where was the place of death of the director of film The Ages Of Lulu?
  COMMIT: la Riera de Gaià

[005/200] qi=5469
Q: Who is Thomas Lloyd-Mostyn's paternal grandfather?
  COMMIT: Edward Lloyd

[006/200] qi=4869
Q: Who is the maternal grandfather of Claudia Antonia?
  COMMIT: Sextus Aelius Catus

[007/200] qi=5358
Q: Wha